# 예금보험공사 RAG 챗봇 최종 크롤링 노트북

이 노트북은 지금까지 확인한 피드백을 하나로 통합한 버전입니다.

## 적용된 기능

- 42개 URL별 최종 수집 정책 자동 적용
- `include_full`, 공개 설명만, 보조 인덱스, 별도 도메인, 링크 전용 구분
- 일반 `<a href>`와 `gfn_openUrl()` 신청·조회·자가진단 버튼 추출
- Action 링크를 `documents.jsonl`의 `actions`로 별도 저장
- BI-004 전체 5페이지와 같은 공개 목록 전체 수집
- FAQ의 모든 페이지를 반복 요청하여 모든 질문·답변 병합
- FAQ가 아니어도 공개 페이지네이션이 있으면 자동 전체 수집
- 요청 페이지와 표시 페이지 불일치·중복 페이지·페이지 누락 검사
- `div`, `span`, `dd` 직접 텍스트 보존
- FAQ `dt` 질문과 `dd` 답변 구조화
- `rowspan`, `colspan` 표 복원
- `gfn_downloadFile()` 첨부파일 버튼 인식
- MT-009 구버전 5천만원 웹툰 설명 비인덱싱
- `검색 초기화`, Adobe Reader 안내, 탭 메뉴 등 UI 노이즈 제거
- 제목만 있는 청크를 다음 문단·표와 병합
- FAQ는 질문+답변, 표는 헤더+행 단위로 청킹
- 중첩된 FAQ·표도 재귀적으로 품질 계산
- 원본 HTML, 페이지별 HTML, Markdown, JSON, documents/chunks, 품질 보고서 생성

## 처리하지 않는 영역

로그인, 본인인증, 개인정보 조회, 신청서 제출은 자동 실행하지 않습니다.

해당 페이지는 공개 설명까지만 수집하고 챗봇에서 이동할 공식 Action 링크만 저장합니다.

## 1. 라이브러리 설치

In [ ]:
%pip -q install "requests>=2.32,<3" "beautifulsoup4>=4.12,<5" "lxml>=5,<7" "pandas>=2.2,<4" "tqdm>=4.66,<5"

## 2. 최종 크롤링 코드 생성

아래 셀은 크롤링 코드를 Colab 런타임에 `kdic_final_pipeline.py`로 생성합니다.

노트북 하나만 보관하면 되며 별도의 Python 파일은 필요하지 않습니다.

In [ ]:
%%writefile kdic_final_pipeline.py
from __future__ import annotations

import hashlib
import json
import mimetypes
import re
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Iterable
from urllib.parse import parse_qs, urlencode, urljoin, urlparse, urlunparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup, Comment, NavigableString, Tag
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

KST = timezone(timedelta(hours=9))


# ============================================================
# 설정
# ============================================================

@dataclass
class PipelineConfig:
    select_business_functions: list[str] | None = None
    run_only_url_ids: list[str] | None = None
    max_urls: int | None = None

    request_delay_seconds: float = 1.2
    request_timeout_seconds: int = 30
    max_retries: int = 3
    max_asset_bytes: int = 50 * 1024 * 1024

    respect_robots_txt: bool = True
    enable_generic_pagination: bool = True
    pagination_max_pages: int = 100
    pagination_page_size: int = 10

    download_direct_attachments: bool = True
    download_images: bool = False
    download_js_attachments_with_playwright: bool = False
    enable_playwright_snapshot: bool = False
    playwright_wait_ms: int = 1500

    create_chunks: bool = True
    chunk_max_chars: int = 1600
    min_chunk_chars: int = 80

    user_agent: str = (
        "KDIC-RAG-Academic-Crawler/1.0 "
        "(low-rate public-document collection; no authentication automation)"
    )

    def __post_init__(self) -> None:
        self.select_business_functions = self.select_business_functions or []
        self.run_only_url_ids = self.run_only_url_ids or []


@dataclass
class OutputPaths:
    root: Path
    raw_html: Path
    raw_assets: Path
    response_meta: Path
    parsed_markdown: Path
    parsed_json: Path
    pagination: Path
    dynamic_diagnostics: Path
    screenshots: Path
    logs: Path
    processed: Path
    manifest: Path
    quality: Path


def create_output_paths(runtime_root: Path) -> OutputPaths:
    run_id = datetime.now(KST).strftime("%Y%m%d_%H%M%S")
    root = runtime_root / f"kdic_crawl_output_{run_id}"
    paths = OutputPaths(
        root=root,
        raw_html=root / "raw" / "html",
        raw_assets=root / "raw" / "assets",
        response_meta=root / "raw" / "response_meta",
        parsed_markdown=root / "parsed" / "markdown",
        parsed_json=root / "parsed" / "json",
        pagination=root / "diagnostics" / "pagination",
        dynamic_diagnostics=root / "diagnostics" / "dynamic",
        screenshots=root / "diagnostics" / "screenshots",
        logs=root / "logs",
        processed=root / "processed",
        manifest=root / "manifest",
        quality=root / "quality",
    )
    for value in asdict(paths).values():
        Path(value).mkdir(parents=True, exist_ok=True)
    return paths


# ============================================================
# 공통 유틸리티
# ============================================================

ATTACHMENT_EXTENSIONS = {
    ".pdf", ".hwp", ".hwpx", ".doc", ".docx", ".xls", ".xlsx",
    ".ppt", ".pptx", ".zip", ".csv", ".txt",
}
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".gif", ".webp", ".svg", ".bmp"}
VIDEO_EXTENSIONS = {".mp4", ".webm", ".mov", ".avi", ".mkv"}

NOISE_SELECTORS = [
    "script", "style", "noscript", "template", ".floatTop", ".floatBottom",
    ".paging", ".pagination", ".pageNavi", ".page_navi", ".sr-only",
    ".skip", ".skipnav", ".skip-navigation", ".loading", ".waitPage",
    "#waitPage", ".pdfViewerGuide", ".adobe-reader", ".tabMenu.clone",
]

NOISE_EXACT_TEXTS = {
    "퀵메뉴", "글자 크기", "글자확대", "글자축소", "KOR", "ENG",
    "상단으로 이동", "검색 초기화", "좌우로 움직여보세요", "표 더보기",
    "계산하기", "닫기", "Adobe Reader 바로가기",
}

NOISE_CONTAINS_TEXTS = {
    "PDF파일 내용이 보이지 않으시면 Adobe Reader를 설치",
    "PDF 파일 내용이 보이지 않으시면 Adobe Reader를 설치",
}

ALLOWED_ACTION_DOMAIN_SUFFIXES = {
    "kdic.or.kr", "fins.kdic.or.kr", "ccrs.or.kr", "scourt.go.kr",
    "klac.or.kr", "law.go.kr", "easylaw.go.kr", "gov.kr",
}

ACTION_KEYWORDS = {
    "신청", "바로가기", "조회", "자가진단", "상담", "진행상황", "접수",
    "신고", "발급", "환급", "위치", "찾아오시는 길", "법령", "규정",
    "다운로드", "전화문의", "문의",
}

DOWNLOAD_CALL_RE = re.compile(
    r"gfn_downloadFile\(\s*['\"]([^'\"]+)['\"]\s*,\s*['\"]([^'\"]+)['\"]\s*\)"
)
URL_IN_JS_PATTERNS = [
    re.compile(r"gfn_openUrl\(\s*['\"]([^'\"]+)['\"]\s*\)"),
    re.compile(r"gfn_moveUrl\(\s*['\"]([^'\"]+)['\"]\s*\)"),
    re.compile(r"window\.open\(\s*['\"]([^'\"]+)['\"]"),
    re.compile(r"(?:window\.)?location(?:\.href)?\s*=\s*['\"]([^'\"]+)['\"]"),
    re.compile(r"location\.replace\(\s*['\"]([^'\"]+)['\"]\s*\)"),
]

BLOCK_TAGS = {
    "div", "section", "article", "aside", "header", "footer", "main",
    "p", "ul", "ol", "dl", "table", "figure", "h1", "h2", "h3",
    "h4", "h5", "h6",
}


def now_kst_iso() -> str:
    return datetime.now(KST).isoformat(timespec="seconds")


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_text(text: str) -> str:
    return sha256_bytes(text.encode("utf-8"))


def norm(text: str | None) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def safe_name(value: str, max_length: int = 100) -> str:
    value = re.sub(r'[\\/:*?"<>|]+', "_", norm(value))
    value = re.sub(r"_+", "_", value).strip("._ ")
    return (value or "untitled")[:max_length]


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, data: Any) -> None:
    ensure_parent(path)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def append_jsonl(path: Path, data: dict[str, Any]) -> None:
    ensure_parent(path)
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(data, ensure_ascii=False) + "\n")


def absolute_url(base_url: str, candidate: str | None, allow_contact: bool = False) -> str | None:
    if not candidate:
        return None
    candidate = candidate.strip()
    if not candidate or candidate.startswith(("#", "javascript:")):
        return None
    if candidate.startswith(("tel:", "mailto:")):
        return candidate if allow_contact else None
    return urljoin(base_url, candidate)


def is_noise_text(text: str) -> bool:
    value = norm(text)
    if not value:
        return True
    if value in NOISE_EXACT_TEXTS:
        return True
    return any(fragment in value for fragment in NOISE_CONTAINS_TEXTS)


def clean_visible_text(text: str) -> str:
    value = norm(text)
    for phrase in ["검색 초기화", "Adobe Reader 바로가기", "계산하기", "닫기"]:
        value = norm(value.replace(phrase, " "))
    for fragment in NOISE_CONTAINS_TEXTS:
        if fragment in value:
            value = norm(value.replace(fragment, " "))
    return value


def row_to_dict(row: pd.Series) -> dict[str, str]:
    return {str(k): "" if pd.isna(v) else str(v) for k, v in row.to_dict().items()}


def unique_dicts(items: Iterable[dict[str, Any]], keys: tuple[str, ...]) -> list[dict[str, Any]]:
    result: list[dict[str, Any]] = []
    seen: set[tuple[Any, ...]] = set()
    for item in items:
        key = tuple(item.get(name) for name in keys)
        if key in seen:
            continue
        seen.add(key)
        result.append(item)
    return result


# ============================================================
# 42개 검토 CSV → 실행 Manifest
# ============================================================

REVIEW_COLUMNS = {
    "문서_ID", "업무_도메인", "목표_도메인", "문서명", "페이지_유형",
    "권장_최종결정", "RAG_본문_인덱싱", "인덱싱_범위", "Action_링크",
    "권장_Action_Type", "Action_인증", "다중페이지_수집정책", "검토_의견",
    "검토_근거", "출처_URL",
}

RUN_DECISIONS = {
    "include_full", "include_reference", "include_full_public", "include_partial",
    "include_partial_action", "include_full_filtered", "include_separate_domain",
    "link_only",
}


def _site_name(url: str) -> str:
    host = (urlparse(url).hostname or "").lower()
    return "금융안심포털" if host.startswith("fins.") else "예금보험공사"


def _page_type(row: pd.Series) -> str:
    doc_id = row["문서_ID"]
    title = row["문서명"]
    policy = row["다중페이지_수집정책"]
    decision = row["권장_최종결정"]
    raw = row.get("페이지_유형", "")
    if decision == "link_only":
        return "link_only"
    if "FAQ" in title.upper() or "FAQ" in policy.upper():
        return "faq"
    if doc_id == "BI-004":
        return "dynamic_lookup"
    if "상호작용" in policy or "인증 서비스" in policy or "신청 서비스" in policy:
        return "interactive_service"
    return raw or "static_page"


def _fetch_strategy(row: pd.Series) -> str:
    decision = row["권장_최종결정"]
    policy = row["다중페이지_수집정책"]
    action = row["Action_링크"]
    if decision == "link_only":
        return "link_only"
    if "필수" in policy or "자동탐지" in policy:
        return "paginated_public"
    if "다운로드" in action:
        return "requests_html_assets"
    return "requests_html"


def _index_mode(value: str) -> str:
    if value == "O":
        return "full"
    if "공개 설명만" in value:
        return "public_only"
    if "보조 인덱스" in value:
        return "reference"
    if "일부 제외" in value:
        return "filtered"
    if "별도 도메인" in value:
        return "separate_domain"
    return "none"


def normalize_review_manifest(review_df: pd.DataFrame) -> pd.DataFrame:
    review_df = review_df.fillna("").astype(str)
    missing = sorted(REVIEW_COLUMNS - set(review_df.columns))
    if missing:
        raise ValueError(f"42개 검토 CSV 필수 열 누락: {missing}")

    records: list[dict[str, str]] = []
    for _, row in review_df.iterrows():
        url = row["출처_URL"]
        decision = row["권장_최종결정"]
        if decision not in RUN_DECISIONS:
            raise ValueError(f"지원하지 않는 결정: {row['문서_ID']}={decision}")
        page_type = _page_type(row)
        action_auth = row["Action_인증"]
        records.append({
            "url_id": row["문서_ID"],
            "business_function": row["업무_도메인"],
            "target_business_function": row["목표_도메인"] or row["업무_도메인"],
            "page_title": row["문서명"],
            "source_url": url,
            "site_name": _site_name(url),
            "normalized_decision": decision,
            "decision_reason": row["검토_의견"],
            "review_basis": row["검토_근거"],
            "page_type": page_type,
            "fetch_strategy": _fetch_strategy(row),
            "parser_profile": "kdic_final_v1",
            "auth_required": action_auth,
            "asset_policy": row["Action_링크"],
            "content_scope": row["인덱싱_범위"],
            "crawl_wave": (
                "META" if decision == "link_only" else
                "W2_DYNAMIC" if ("필수" in row["다중페이지_수집정책"] or row["문서_ID"] == "BI-004") else
                "W1_ASSET" if "다운로드" in row["Action_링크"] else
                "W1_STATIC"
            ),
            "rag_index_mode": _index_mode(row["RAG_본문_인덱싱"]),
            "rag_index_label": row["RAG_본문_인덱싱"],
            "action_policy": row["Action_링크"],
            "expected_action_types": row["권장_Action_Type"],
            "action_auth": action_auth,
            "pagination_policy": row["다중페이지_수집정책"],
        })

    manifest = pd.DataFrame(records)
    if manifest["url_id"].duplicated().any():
        raise ValueError("Manifest url_id 중복")
    if len(manifest) != 42:
        raise ValueError(f"검토표는 42개여야 합니다. 현재 {len(manifest)}개")
    return manifest


# ============================================================
# HTTP 수집
# ============================================================

@dataclass
class FetchResult:
    requested_url: str
    final_url: str
    status_code: int
    ok: bool
    content_type: str
    encoding: str
    elapsed_seconds: float
    collected_at: str
    content_length: int
    raw_sha256: str
    redirect_history: list[dict[str, Any]]
    selected_headers: dict[str, str]
    body: bytes


def create_session(config: PipelineConfig) -> requests.Session:
    retry = Retry(
        total=config.max_retries,
        connect=config.max_retries,
        read=config.max_retries,
        status=config.max_retries,
        backoff_factor=0.8,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET", "POST", "HEAD"}),
        respect_retry_after_header=True,
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=2, pool_maxsize=2)
    session = requests.Session()
    session.headers.update({
        "User-Agent": config.user_agent,
        "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.5",
    })
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def choose_encoding(response: requests.Response) -> str:
    encoding = response.encoding
    if not encoding or encoding.lower() in {"iso-8859-1", "ascii"}:
        encoding = response.apparent_encoding or "utf-8"
    return encoding


def robots_allows(url: str, config: PipelineConfig) -> tuple[bool, str]:
    if not config.respect_robots_txt:
        return True, "disabled"
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    try:
        parser = RobotFileParser()
        parser.set_url(robots_url)
        parser.read()
        allowed = parser.can_fetch(config.user_agent, url)
        return allowed, "allowed" if allowed else "disallowed"
    except Exception as error:
        return True, f"unavailable:{type(error).__name__}"


def response_to_fetch_result(response: requests.Response, requested_url: str, elapsed: float) -> FetchResult:
    encoding = choose_encoding(response)
    response.encoding = encoding
    selected = {
        key: value for key, value in response.headers.items()
        if key in {"Content-Type", "Content-Length", "Last-Modified", "ETag", "Cache-Control", "Content-Disposition"}
    }
    return FetchResult(
        requested_url=requested_url,
        final_url=response.url,
        status_code=response.status_code,
        ok=response.ok,
        content_type=response.headers.get("Content-Type", ""),
        encoding=encoding,
        elapsed_seconds=round(elapsed, 3),
        collected_at=now_kst_iso(),
        content_length=len(response.content),
        raw_sha256=sha256_bytes(response.content),
        redirect_history=[
            {"status_code": item.status_code, "url": item.url, "location": item.headers.get("Location")}
            for item in response.history
        ],
        selected_headers=selected,
        body=response.content,
    )


def fetch_url(session: requests.Session, url: str, config: PipelineConfig) -> FetchResult:
    allowed, status = robots_allows(url, config)
    if not allowed:
        raise PermissionError(f"robots.txt에서 허용하지 않음: {url}")
    started = time.perf_counter()
    response = session.get(url, timeout=config.request_timeout_seconds, allow_redirects=True)
    result = response_to_fetch_result(response, url, time.perf_counter() - started)
    result.selected_headers["Robots-Check"] = status
    return result


def save_fetch_result(paths: OutputPaths, row: dict[str, str], result: FetchResult) -> tuple[Path, Path]:
    folder = safe_name(row["business_function"])
    html_path = paths.raw_html / folder / f"{row['url_id']}.html"
    meta_path = paths.response_meta / folder / f"{row['url_id']}.json"
    ensure_parent(html_path)
    html_path.write_bytes(result.body)
    meta = asdict(result)
    meta.pop("body", None)
    meta.update({
        "url_id": row["url_id"],
        "business_function": row["business_function"],
        "page_title_manifest": row["page_title"],
        "fetch_strategy": row["fetch_strategy"],
        "parser_profile": row["parser_profile"],
        "raw_html_path": str(html_path.relative_to(paths.root)),
    })
    write_json(meta_path, meta)
    return html_path, meta_path


# ============================================================
# 결정론적 HTML 파싱
# ============================================================

def safe_text(node: Tag) -> str:
    fragment = BeautifulSoup(str(node), "lxml")
    root = fragment.body.find() if fragment.body else fragment.find()
    if not root:
        return ""
    for child in root.select(
        ".sr-only,.hide,script,style,noscript,template,.floatTop,.floatBottom,.paging,.pagination"
    ):
        child.decompose()
    for image in root.find_all("img"):
        image.replace_with(norm(image.get("alt", "")))
    for br in root.find_all("br"):
        br.replace_with(" ")
    return clean_visible_text(root.get_text(" ", strip=True))


def infer_file_format(node: Tag) -> str:
    text = (" ".join(node.get("class", [])).lower() + " " + norm(node.get_text(" ", strip=True)).lower())
    for keyword, fmt in [
        ("icohwp", "HWP"), ("hwp", "HWP"), ("icopdf", "PDF"), ("pdf", "PDF"),
        ("xlsx", "XLSX"), ("excel", "XLSX"), ("docx", "DOCX"), ("doc", "DOC"),
    ]:
        if keyword in text:
            return fmt
    return "FILE"


def cell_text(cell: Tag) -> str:
    fragment = BeautifulSoup(str(cell), "lxml")
    root = fragment.body.find() if fragment.body else fragment.find()
    if not root:
        return ""
    for child in root.select("script,style,noscript,template,.sr-only"):
        child.decompose()
    for button in root.find_all("button"):
        fmt = infer_file_format(button)
        label = re.sub(r"다운로드$", "", norm(button.get_text(" ", strip=True))).strip()
        button.replace_with(f"{label} {fmt} 다운로드".strip())
    for image in root.find_all("img"):
        image.replace_with(norm(image.get("alt", "")))
    for br in root.find_all("br"):
        br.replace_with(" ")
    return norm(root.get_text(" ", strip=True))


def expand_table(table: Tag) -> tuple[list[str], list[list[str]]]:
    grid: list[list[str]] = []
    active: dict[int, list[Any]] = {}
    header_flags: list[bool] = []
    max_columns = 0

    for tr in table.find_all("tr"):
        row: list[str] = []
        column = 0
        cells = tr.find_all(["th", "td"], recursive=False)

        def fill_active() -> None:
            nonlocal column
            while column in active:
                remaining, value = active[column]
                while len(row) <= column:
                    row.append("")
                row[column] = value
                remaining -= 1
                if remaining <= 0:
                    del active[column]
                else:
                    active[column] = [remaining, value]
                column += 1

        fill_active()
        header_flags.append(any(cell.name == "th" for cell in cells))
        for cell in cells:
            fill_active()
            text = cell_text(cell)
            rowspan = max(1, int(cell.get("rowspan", 1) or 1))
            colspan = max(1, int(cell.get("colspan", 1) or 1))
            for offset in range(colspan):
                target = column + offset
                while len(row) <= target:
                    row.append("")
                row[target] = text
                if rowspan > 1:
                    active[target] = [rowspan - 1, text]
            column += colspan
        if active:
            final_col = max(active)
            while column <= final_col:
                if column in active:
                    fill_active()
                else:
                    while len(row) <= column:
                        row.append("")
                    column += 1
        max_columns = max(max_columns, len(row))
        if row and any(row):
            grid.append(row)

    if not grid:
        return [], []
    for row in grid:
        row.extend([""] * (max_columns - len(row)))
    if table.thead:
        header_count = len(table.thead.find_all("tr", recursive=False))
    else:
        header_count = 1 if header_flags and header_flags[0] else 0
    header_count = max(1, header_count)
    header_rows = grid[:header_count]
    rows = grid[header_count:]
    headers: list[str] = []
    for col in range(max_columns):
        values: list[str] = []
        for hrow in header_rows:
            value = norm(hrow[col])
            if value and value not in values:
                values.append(value)
        headers.append(" / ".join(values) if values else f"열{col + 1}")
    counts: dict[str, int] = {}
    unique_headers: list[str] = []
    for header in headers:
        counts[header] = counts.get(header, 0) + 1
        unique_headers.append(header if counts[header] == 1 else f"{header} {counts[header]}")
    return unique_headers, rows


def extract_attachments(root: Tag, base_url: str) -> list[dict[str, Any]]:
    attachments: list[dict[str, Any]] = []
    seen: set[Any] = set()
    for button in root.find_all(["button", "a"], onclick=True):
        match = DOWNLOAD_CALL_RE.search(button.get("onclick", ""))
        if not match:
            continue
        key = (match.group(1), match.group(2))
        if key in seen:
            continue
        seen.add(key)
        fmt = infer_file_format(button)
        label = re.sub(r"다운로드$", "", norm(button.get_text(" ", strip=True))).strip()
        row_context = ""
        parent_row = button.find_parent("tr")
        if parent_row:
            values = []
            for cell in parent_row.find_all(["th", "td"], recursive=False):
                if button in cell.descendants:
                    continue
                value = cell_text(cell)
                if value and "다운로드" not in value and value not in values:
                    values.append(value)
            row_context = " | ".join(values[:4])
        display_name = label or row_context or f"{fmt} 첨부파일"
        if fmt not in display_name.upper():
            display_name = f"{display_name} ({fmt})"
        attachments.append({
            "attachment_id": sha256_text(f"{match.group(1)}|{match.group(2)}")[:16],
            "display_name": display_name,
            "file_format": fmt,
            "download_method": "gfn_downloadFile",
            "token1": match.group(1),
            "token2": match.group(2),
            "row_context": row_context,
            "button_text": norm(button.get_text(" ", strip=True)),
            "source_url": base_url,
            "download_status": "metadata_only",
        })

    for anchor in root.find_all("a", href=True):
        url = absolute_url(base_url, anchor.get("href"))
        if not url:
            continue
        ext = Path(urlparse(url).path).suffix.lower()
        if ext not in ATTACHMENT_EXTENSIONS:
            continue
        key = ("href", url)
        if key in seen:
            continue
        seen.add(key)
        attachments.append({
            "attachment_id": sha256_text(url)[:16],
            "display_name": norm(anchor.get_text(" ", strip=True)) or Path(urlparse(url).path).name,
            "file_format": ext.lstrip(".").upper(),
            "download_method": "href",
            "url": url,
            "source_url": base_url,
            "download_status": "metadata_only",
        })
    return attachments


def classify_action_type(label: str, expected_types: str = "") -> str:
    text = norm(label)
    tests = [
        ("자가진단", "self_check"), ("진행상황", "progress"), ("신고", "report"),
        ("상담", "consult"), ("부채증명", "issue"), ("발급", "issue"),
        ("환급", "refund"), ("법령", "legal_reference"), ("규정", "legal_reference"),
        ("위치", "location"), ("찾아오시는", "location"), ("전화", "contact"),
        ("문의", "contact"), ("조회", "lookup"), ("다운로드", "download"),
        ("신청", "apply"), ("접수", "apply"),
    ]
    for keyword, action_type in tests:
        if keyword in text:
            return action_type
    return "related_service"


def allowed_action_domain(url: str) -> bool:
    if url.startswith(("tel:", "mailto:")):
        return True
    host = (urlparse(url).hostname or "").lower()
    return any(host == suffix or host.endswith("." + suffix) for suffix in ALLOWED_ACTION_DOMAIN_SUFFIXES)


def _extract_js_url(onclick: str) -> str | None:
    for pattern in URL_IN_JS_PATTERNS:
        match = pattern.search(onclick or "")
        if match:
            return match.group(1)
    direct = re.search(r"https?://[^'\"\s)]+", onclick or "")
    return direct.group(0) if direct else None


def action_label_with_context(node: Tag, label: str) -> str:
    label = norm(label)
    if label not in {"바로가기", "신청", "조회", "다운로드"}:
        return label
    parent = node.find_parent(["div", "li", "td", "section", "article"])
    if parent:
        context_node = parent.find(["strong", "h2", "h3", "h4", "dt"], recursive=True)
        context = norm(context_node.get_text(" ", strip=True)) if context_node else ""
        if context and context != label:
            return f"{context} {label}"
    previous = node.find_previous(["strong", "h2", "h3", "h4", "dt"])
    context = norm(previous.get_text(" ", strip=True)) if previous else ""
    return f"{context} {label}".strip() if context else label


def extract_actions(
    root: Tag,
    base_url: str,
    manifest_row: dict[str, str],
    attachments: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    actions: list[dict[str, Any]] = []
    expected = manifest_row.get("expected_action_types", "")
    expected_set = {value.strip() for value in expected.split("|") if value.strip()}
    action_policy = manifest_row.get("action_policy", "")
    if action_policy.startswith("X"):
        return []
    seen: set[tuple[str, str]] = set()

    def add(label: str, url: str, source_tag: str, js_function: str = "") -> None:
        label = norm(label)
        if not label or is_noise_text(label):
            return
        if any(noise in label for noise in ["디지털역사관", "예솜이", "챗봇", "앱 설치", "공식 홈페이지"]):
            return
        action_like = any(keyword in label for keyword in ACTION_KEYWORDS)
        if not action_like:
            return
        action_type = classify_action_type(label, expected)
        host = (urlparse(url).hostname or "").lower()
        if host == "law.go.kr" or host.endswith(".law.go.kr"):
            action_type = "legal_reference"
        if expected_set and action_type not in expected_set and action_type not in {"download", "legal_reference", "location", "contact"}:
            return
        key = (url, label)
        if key in seen:
            return
        seen.add(key)
        actions.append({
            "action_id": sha256_text(f"{manifest_row['url_id']}|{url}|{label}")[:16],
            "label": label,
            "url": url,
            "action_type": action_type,
            "source_tag": source_tag,
            "javascript_function": js_function,
            "auth_required": manifest_row.get("action_auth", manifest_row.get("auth_required", "")),
            "official_domain": allowed_action_domain(url),
            "requires_review": not allowed_action_domain(url),
        })

    for anchor in root.find_all("a", href=True):
        href = anchor.get("href", "").strip()
        url = absolute_url(base_url, href, allow_contact=True)
        if not url:
            continue
        label = norm(anchor.get_text(" ", strip=True)) or norm(anchor.get("title", ""))
        label = action_label_with_context(anchor, label)
        if any(keyword in label for keyword in ACTION_KEYWORDS):
            add(label, url, "a")

    for node in root.find_all(["button", "a", "input"], onclick=True):
        onclick = node.get("onclick", "")
        candidate = _extract_js_url(onclick)
        if not candidate:
            continue
        url = absolute_url(base_url, candidate, allow_contact=True)
        if not url:
            continue
        label = norm(node.get_text(" ", strip=True)) or norm(node.get("value", "")) or norm(node.get("title", ""))
        label = action_label_with_context(node, label)
        function_name = onclick.split("(", 1)[0].strip()
        add(label, url, node.name, function_name)

    # 다운로드도 챗봇 Action으로 제공할 수 있도록 연결합니다.
    for attachment in attachments:
        url = attachment.get("url") or base_url
        label = attachment.get("display_name", "첨부파일 다운로드")
        key = (url, label)
        if key in seen:
            continue
        seen.add(key)
        actions.append({
            "action_id": attachment["attachment_id"],
            "label": label,
            "url": url,
            "action_type": "download",
            "source_tag": "attachment",
            "attachment_id": attachment["attachment_id"],
            "download_method": attachment["download_method"],
            "auth_required": "불필요",
            "official_domain": True,
            "requires_review": False,
        })

    return actions


def extract_links(root: Tag, base_url: str) -> list[dict[str, Any]]:
    links: list[dict[str, Any]] = []
    seen: set[tuple[str, str]] = set()
    for anchor in root.find_all("a", href=True):
        url = absolute_url(base_url, anchor.get("href"))
        text = norm(anchor.get_text(" ", strip=True))
        if not url or not text or is_noise_text(text):
            continue
        key = (url, text)
        if key in seen:
            continue
        seen.add(key)
        links.append({"url": url, "anchor_text": text})
    return links


def extract_images(root: Tag, base_url: str) -> list[dict[str, Any]]:
    images: list[dict[str, Any]] = []
    seen: set[str] = set()
    for image in root.find_all("img"):
        source = image.get("src") or image.get("data-src") or image.get("data-original")
        url = absolute_url(base_url, source)
        if not url or url in seen:
            continue
        seen.add(url)
        alt = norm(image.get("alt", ""))
        filename = Path(urlparse(url).path).name
        lowered = filename.lower()
        role = "decorative"
        if "webtoon" in lowered:
            role = "supplementary_visual"
        elif any(keyword in lowered for keyword in ["proc", "process", "step", "flow"]):
            role = "process_diagram"
        elif alt:
            role = "content_image"
        images.append({"url": url, "alt": alt, "filename": filename, "image_role": role})
    return images


class StructureParser:
    def __init__(self, page_type: str = "static_page"):
        self.blocks: list[dict[str, Any]] = []
        self.page_type = page_type

    def add(self, block: dict[str, Any] | None) -> None:
        if not block:
            return
        if self.blocks and block.get("type") in {"paragraph", "heading"}:
            if self.blocks[-1].get("type") == block.get("type") and self.blocks[-1].get("text") == block.get("text"):
                return
        self.blocks.append(block)

    def parse(self, root: Tag) -> list[dict[str, Any]]:
        faq_dls = root.select(".accoWrap .accodian > dl, .accodian.con.ico > dl, .accordion > dl")
        if self.page_type == "faq" or faq_dls:
            if not faq_dls:
                faq_dls = [dl for dl in root.find_all("dl") if dl.find("dt", recursive=False) and dl.find("dd", recursive=False)]
            faq_ids = {id(node) for node in faq_dls}
            for child in root.find_all(recursive=False):
                if isinstance(child, Tag) and any(id(dl) in faq_ids for dl in child.select("dl")):
                    continue
                self.process_node(child)
            for dl in faq_dls:
                self.parse_faq(dl)
        else:
            for child in root.find_all(recursive=False):
                self.process_node(child)
        return self.blocks

    def parse_faq(self, dl: Tag) -> None:
        dt = dl.find("dt", recursive=False)
        dd = dl.find("dd", recursive=False)
        if not dt or not dd:
            return
        question = re.sub(r"^\s*질문\s*", "", safe_text(dt))
        question = re.sub(r"\s*열기\s*$", "", question)
        answer_parser = StructureParser("static_page")
        answer_parser.process_node(dd)
        if not answer_parser.blocks:
            answer_parser.add({"type": "paragraph", "text": re.sub(r"^\s*답변\s*", "", safe_text(dd))})
        answer_text = blocks_text(answer_parser.blocks)
        if question and answer_text:
            self.add({
                "type": "faq", "question": norm(question),
                "answer_blocks": answer_parser.blocks, "answer_text": answer_text,
            })

    def parse_definition_list(self, dl: Tag) -> None:
        children = [child for child in dl.find_all(recursive=False) if isinstance(child, Tag)]
        index = 0
        while index < len(children):
            if children[index].name != "dt":
                self.process_node(children[index])
                index += 1
                continue
            dt = children[index]
            dd = children[index + 1] if index + 1 < len(children) and children[index + 1].name == "dd" else None
            term = re.sub(r"\s*열기\s*$", "", safe_text(dt))
            term = re.sub(r"^\s*\d{1,2}\s+(?=\D)", "", term)
            if dd:
                parser = StructureParser("static_page")
                parser.process_node(dd)
                if not parser.blocks:
                    parser.add({"type": "paragraph", "text": safe_text(dd)})
                self.add({
                    "type": "definition", "term": norm(term),
                    "definition_blocks": parser.blocks,
                    "definition_text": blocks_text(parser.blocks),
                })
                index += 2
            else:
                self.add({"type": "heading", "level": 3, "text": norm(term)})
                index += 1

    def process_node(self, node: Any) -> None:
        if isinstance(node, Comment):
            return
        if isinstance(node, NavigableString):
            text = norm(str(node))
            if text and not is_noise_text(text):
                self.add({"type": "paragraph", "text": text})
            return
        if not isinstance(node, Tag):
            return
        name = node.name.lower()
        classes = set(node.get("class", []))
        if name in {"script", "style", "noscript", "template"} or classes & {"floatTop", "floatBottom", "paging", "pagination", "sr-only"}:
            return
        if name in {"button", "input"}:
            return
        if name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
            text = safe_text(node)
            if text and not is_noise_text(text):
                self.add({"type": "heading", "level": int(name[1]), "text": text})
            return
        if name == "p":
            text = safe_text(node)
            if text and not is_noise_text(text):
                self.add({"type": "paragraph", "text": text})
            return
        if name == "table":
            headers, rows = expand_table(node)
            if headers or rows:
                self.add({"type": "table", "headers": headers, "rows": rows, "row_count": len(rows)})
            return
        if name in {"ul", "ol"}:
            items: list[str] = []
            for li in node.find_all("li", recursive=False):
                fragment = BeautifulSoup(str(li), "lxml")
                copied = fragment.body.find("li") if fragment.body else None
                if not copied:
                    continue
                for nested in copied.find_all(["ul", "ol"]):
                    nested.decompose()
                text = safe_text(copied)
                if text and not is_noise_text(text):
                    items.append(text)
            # 단순 탭 메뉴는 제거합니다.
            if set(items).issubset({"착오송금인", "착오송금수취인"}):
                items = []
            if items:
                self.add({"type": "list", "ordered": name == "ol", "items": items})
            for li in node.find_all("li", recursive=False):
                for nested in li.find_all(["ul", "ol"], recursive=False):
                    self.process_node(nested)
            return
        if name == "dl":
            self.parse_definition_list(node)
            return
        if name == "figure":
            caption = node.find("figcaption")
            if caption:
                text = safe_text(caption)
                if text and not is_noise_text(text):
                    self.add({"type": "paragraph", "text": text})
            return

        inline: list[str] = []
        def flush() -> None:
            nonlocal inline
            text = norm(" ".join(inline))
            inline = []
            if text and not is_noise_text(text):
                self.add({"type": "paragraph", "text": text})

        for child in node.children:
            if isinstance(child, Comment):
                continue
            if isinstance(child, NavigableString):
                value = norm(str(child))
                if value:
                    inline.append(value)
            elif isinstance(child, Tag):
                if child.name.lower() == "br":
                    inline.append(" ")
                elif child.name.lower() in BLOCK_TAGS:
                    flush()
                    self.process_node(child)
                elif child.name.lower() in {"button", "input"}:
                    continue
                else:
                    value = safe_text(child)
                    if value:
                        inline.append(value)
        flush()


def render_blocks(blocks: list[dict[str, Any]]) -> str:
    lines: list[str] = []
    for block in blocks:
        kind = block.get("type")
        if kind == "heading":
            level = min(max(int(block.get("level", 2)), 2), 6)
            lines.append(f"{'#' * level} {block.get('text', '')}")
        elif kind == "paragraph":
            lines.append(block.get("text", ""))
        elif kind == "list":
            for index, item in enumerate(block.get("items", []), start=1):
                prefix = f"{index}. " if block.get("ordered") else "- "
                lines.append(prefix + item)
        elif kind == "definition":
            lines.append("### " + block.get("term", ""))
            lines.append(render_blocks(block.get("definition_blocks", [])))
        elif kind == "faq":
            lines.append("### Q. " + block.get("question", ""))
            lines.append(render_blocks(block.get("answer_blocks", [])))
        elif kind == "table":
            headers = [norm(value).replace("|", "\\|") for value in block.get("headers", [])]
            if headers:
                lines.append("| " + " | ".join(headers) + " |")
                lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
                for row in block.get("rows", []):
                    values = [norm(value).replace("|", "\\|") for value in row]
                    values.extend([""] * (len(headers) - len(values)))
                    lines.append("| " + " | ".join(values[:len(headers)]) + " |")
        lines.append("")
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def blocks_text(blocks: list[dict[str, Any]]) -> str:
    values: list[str] = []
    for block in blocks:
        kind = block.get("type")
        if kind in {"heading", "paragraph"}:
            values.append(block.get("text", ""))
        elif kind == "list":
            values.extend(block.get("items", []))
        elif kind == "definition":
            values.extend([block.get("term", ""), block.get("definition_text", "")])
        elif kind == "faq":
            values.extend([block.get("question", ""), block.get("answer_text", "")])
        elif kind == "table":
            values.extend(block.get("headers", []))
            for row in block.get("rows", []):
                values.extend(row)
    return norm(" ".join(values))


def iter_blocks_recursive(blocks: list[dict[str, Any]]) -> Iterable[dict[str, Any]]:
    for block in blocks:
        yield block
        for key in ("definition_blocks", "answer_blocks"):
            children = block.get(key, [])
            if isinstance(children, list):
                yield from iter_blocks_recursive(children)


def count_block_type(blocks: list[dict[str, Any]], target: str) -> int:
    return sum(1 for block in iter_blocks_recursive(blocks) if block.get("type") == target)


def block_fingerprint(block: dict[str, Any]) -> str:
    return sha256_text(json.dumps(block, ensure_ascii=False, sort_keys=True))


def apply_document_filters(document: dict[str, Any]) -> None:
    excluded: list[dict[str, Any]] = []
    cleaned: list[dict[str, Any]] = []
    for block in document.get("blocks", []):
        text = blocks_text([block])
        if document["doc_id"] == "MT-009" and any(token in text for token in ["5천만 원", "5천만원", "1천만원 이하", "1천만원 초과"]):
            excluded.append({"reason": "구버전 지원금액이 포함된 웹툰 설명", "content": text})
            continue
        if is_noise_text(text):
            continue
        cleaned.append(block)
    document["blocks"] = cleaned
    if excluded:
        document["excluded_content"] = excluded
        document["content_filter"] = "legacy_amount_removed"


def refresh_document(document: dict[str, Any]) -> None:
    document["content_markdown"] = render_blocks(document.get("blocks", []))
    document["content_text"] = blocks_text(document.get("blocks", []))
    parsed_hash = sha256_text(document["content_markdown"])
    document["parsed_content_sha256"] = parsed_hash
    document["content_hash"] = parsed_hash


def select_content_root(soup: BeautifulSoup) -> Tag:
    root = (
        soup.select_one(".contents") or soup.select_one("#contents") or
        soup.select_one("#content") or soup.select_one("main") or soup.body
    )
    if not root:
        raise ValueError("본문 루트를 찾지 못했습니다.")
    return root


def parse_html_document(
    html_bytes: bytes,
    final_url: str,
    manifest_row: dict[str, str],
    encoding: str,
) -> dict[str, Any]:
    soup = BeautifulSoup(html_bytes.decode(encoding, errors="replace"), "lxml")
    root = select_content_root(soup)

    attachments = extract_attachments(root, final_url)
    actions = extract_actions(root, final_url, manifest_row, attachments)
    if not actions and manifest_row.get("page_type") == "dynamic_lookup":
        actions = [{
            "action_id": sha256_text(f"{manifest_row['url_id']}|{manifest_row['source_url']}|lookup")[:16],
            "label": f"{manifest_row['page_title']} 조회 바로가기",
            "url": manifest_row["source_url"],
            "action_type": "lookup",
            "source_tag": "source_page",
            "auth_required": manifest_row.get("action_auth", "불필요"),
            "official_domain": True,
            "requires_review": False,
        }]
    images = extract_images(root, final_url)
    links = extract_links(root, final_url)

    source_fragment = BeautifulSoup(str(root), "lxml")
    source_root = source_fragment.body.find() if source_fragment.body else source_fragment.find()
    if source_root:
        for selector in NOISE_SELECTORS:
            for node in source_root.select(selector):
                node.decompose()
        source_text = norm(source_root.get_text(" ", strip=True))
    else:
        source_text = ""

    for selector in NOISE_SELECTORS:
        for node in root.select(selector):
            node.decompose()

    parser = StructureParser(manifest_row.get("page_type", "static_page"))
    blocks = parser.parse(root)
    document = {
        "doc_id": manifest_row["url_id"],
        "parent_doc_id": manifest_row["url_id"],
        "record_type": "page",
        "indexable": manifest_row.get("rag_index_mode") != "none",
        "rag_index_mode": manifest_row.get("rag_index_mode", "full"),
        "title": manifest_row["page_title"],
        "manifest_title": manifest_row["page_title"],
        "html_title": norm(soup.title.get_text(" ", strip=True)) if soup.title else "",
        "source_url": manifest_row["source_url"],
        "final_url": final_url,
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get("target_business_function", manifest_row["business_function"]),
        "page_type": manifest_row["page_type"],
        "parser_profile": manifest_row["parser_profile"],
        "breadcrumb": [manifest_row["business_function"], manifest_row["page_title"]],
        "blocks": blocks,
        "links": links,
        "actions": actions,
        "attachments": attachments,
        "images": images,
        "videos": [],
        "policy": {
            "normalized_decision": manifest_row["normalized_decision"],
            "content_scope": manifest_row["content_scope"],
            "rag_index_label": manifest_row["rag_index_label"],
            "action_policy": manifest_row["action_policy"],
            "expected_action_types": manifest_row["expected_action_types"],
            "pagination_policy": manifest_row["pagination_policy"],
            "auth_required": manifest_row["auth_required"],
        },
        "parsed_at": now_kst_iso(),
    }
    apply_document_filters(document)
    refresh_document(document)
    document["quality"] = build_quality(document, source_text, manifest_row)
    return document


def build_quality(document: dict[str, Any], source_text: str, row: dict[str, str]) -> dict[str, Any]:
    blocks = document.get("blocks", [])
    content = document.get("content_text", "")
    faq_count = count_block_type(blocks, "faq")
    table_count = count_block_type(blocks, "table")
    markdown_lines = {norm(line.lstrip("#-0123456789. ")) for line in document.get("content_markdown", "").splitlines()}
    noise_hits = [value for value in NOISE_EXACT_TEXTS if value in markdown_lines]
    reasons: list[str] = []
    if document.get("indexable", True) and len(content) < 80:
        reasons.append("본문이 80자 미만")
    retention = round(len(content) / max(1, len(source_text)), 3)
    if document.get("indexable", True) and retention < 0.55:
        reasons.append("본문 보존율 55% 미만")
    if noise_hits:
        reasons.append("공통 UI 문구 잔존")
    if row.get("page_type") == "faq" and faq_count == 0:
        reasons.append("FAQ 질문-답변 추출 실패")
    if row.get("action_policy", "").startswith("O") and not document.get("actions"):
        reasons.append("예상 Action 링크 미검출")
    status = "fail" if any(value in reasons for value in ["공통 UI 문구 잔존", "FAQ 질문-답변 추출 실패"]) else ("review" if reasons else "pass")
    return {
        "status": status,
        "reasons": reasons,
        "source_text_chars": len(source_text),
        "parsed_text_chars": len(content),
        "retention_ratio": retention,
        "faq_count": faq_count,
        "table_count": table_count,
        "attachment_button_count": len(document.get("attachments", [])),
        "action_count": len(document.get("actions", [])),
        "noise_hits": noise_hits,
    }


# ============================================================
# 공통 페이지네이션 수집
# ============================================================

@dataclass
class PaginationPlan:
    method: str
    endpoint: str
    page_param: str
    page_size_param: str | None
    page_size: int
    last_internal_index: int
    index_base: int
    base_payload: dict[str, str]
    detection_method: str


def extract_form_payload(form: Tag | None) -> dict[str, str]:
    payload: dict[str, str] = {}
    if not form:
        return payload
    for field in form.select("input[name], select[name], textarea[name]"):
        name = norm(field.get("name", ""))
        if not name:
            continue
        if field.name == "select":
            selected = field.find("option", selected=True) or field.find("option")
            value = selected.get("value", "") if selected else ""
        elif field.name == "textarea":
            value = field.get_text("", strip=False)
        else:
            field_type = norm(field.get("type", "text")).lower()
            if field_type in {"checkbox", "radio"} and not field.has_attr("checked"):
                continue
            value = field.get("value", "")
        payload[name] = str(value or "")
    return payload


def extract_script_text(soup: BeautifulSoup) -> str:
    return "\n".join(script.get_text("\n", strip=True) for script in soup.find_all("script") if not script.get("src"))


def detect_pagination_plan(
    html_bytes: bytes,
    encoding: str,
    final_url: str,
    row: dict[str, str],
    config: PipelineConfig,
) -> PaginationPlan | None:
    if not config.enable_generic_pagination:
        return None
    policy = row.get("pagination_policy", "")
    if "제외" in policy:
        return None

    soup = BeautifulSoup(html_bytes.decode(encoding, errors="replace"), "lxml")
    onclick_indexes: list[int] = []
    for node in soup.select(".paging [onclick*='chgPagingNo'], .pagination [onclick*='chgPagingNo']"):
        match = re.search(r"chgPagingNo\(\s*(\d+)\s*\)", node.get("onclick", ""))
        if match:
            onclick_indexes.append(int(match.group(1)))

    if onclick_indexes:
        last_index = max(onclick_indexes)
        if last_index <= 0:
            return None
        form = soup.select_one("form#srchForm") or soup.select_one("form[name='srchForm']")
        payload = extract_form_payload(form)
        scripts = extract_script_text(soup)
        action = norm(form.get("action", "")) if form else ""
        if not action:
            match = re.search(r"[\"']#srchForm[\"']\)\.attr\(\s*[\"']action[\"']\s*,\s*[\"']([^\"']+)[\"']", scripts)
            if not match:
                match = re.search(r"attr\(\s*[\"']action[\"']\s*,\s*[\"']([^\"']+)[\"']", scripts)
            action = match.group(1) if match else ""
        endpoint = urljoin(final_url, action) if action else final_url
        page_param = "curPage"
        for candidate in ["curPage", "pageIndex", "pageNo", "currentPage", "page"]:
            if candidate in payload or re.search(rf"name\s*[:=]\s*['\"]{candidate}['\"]", scripts):
                page_param = candidate
                break
        page_size_param = "pageSize" if "pageSize" in payload or "pageSize" in scripts else None
        return PaginationPlan(
            method="POST", endpoint=endpoint, page_param=page_param,
            page_size_param=page_size_param, page_size=config.pagination_page_size,
            last_internal_index=last_index, index_base=0, base_payload=payload,
            detection_method="chgPagingNo",
        )

    # query string 기반 페이지 링크도 지원합니다.
    page_links: list[tuple[str, str, int]] = []
    for anchor in soup.select(".paging a[href], .pagination a[href]"):
        href = anchor.get("href", "")
        if not href or href.startswith("javascript:"):
            continue
        absolute = urljoin(final_url, href)
        query = parse_qs(urlparse(absolute).query)
        for param in ["curPage", "pageIndex", "pageNo", "currentPage", "page"]:
            if param in query and query[param] and str(query[param][0]).isdigit():
                page_links.append((absolute, param, int(query[param][0])))
    if page_links:
        _, param, max_page = max(page_links, key=lambda item: item[2])
        if max_page <= 1:
            return None
        return PaginationPlan(
            method="GET", endpoint=final_url, page_param=param, page_size_param=None,
            page_size=config.pagination_page_size, last_internal_index=max_page,
            index_base=1, base_payload={}, detection_method="query_link",
        )
    return None


def displayed_page_number(html_bytes: bytes, encoding: str) -> int | None:
    soup = BeautifulSoup(html_bytes.decode(encoding, errors="replace"), "lxml")
    current = soup.select_one(".paging strong[title*='현재 페이지'] span") or soup.select_one(".pagination .active")
    if not current:
        return None
    text = norm(current.get_text(" ", strip=True))
    return int(text) if text.isdigit() else None


def repeatable_signature(document: dict[str, Any]) -> str:
    selected = [
        block for block in document.get("blocks", [])
        if block.get("type") in {"faq", "table", "list", "definition"}
    ]
    if not selected:
        selected = document.get("blocks", [])
    return sha256_text(json.dumps(selected, ensure_ascii=False, sort_keys=True))


def merge_table_blocks(target: dict[str, Any], incoming: dict[str, Any]) -> None:
    existing = {tuple(row) for row in target.get("rows", [])}
    for row in incoming.get("rows", []):
        key = tuple(row)
        if key not in existing:
            existing.add(key)
            target.setdefault("rows", []).append(row)
    target["row_count"] = len(target.get("rows", []))


def merge_paginated_documents(documents: list[dict[str, Any]], row: dict[str, str]) -> dict[str, Any]:
    merged = documents[0]
    page_type = row.get("page_type", "")

    if page_type == "faq":
        base_non_faq = [block for block in merged["blocks"] if block.get("type") != "faq"]
        faq_blocks: list[dict[str, Any]] = []
        seen: set[tuple[str, str]] = set()
        for document in documents:
            for block in document.get("blocks", []):
                if block.get("type") != "faq":
                    continue
                key = (norm(block.get("question", "")), norm(block.get("answer_text", "")))
                if key in seen:
                    continue
                seen.add(key)
                faq_blocks.append(block)
        merged["blocks"] = base_non_faq + faq_blocks
    else:
        # 같은 헤더를 가진 표는 행을 합치고, 나머지 새 블록은 중복 없이 추가합니다.
        table_by_headers: dict[tuple[str, ...], dict[str, Any]] = {
            tuple(block.get("headers", [])): block
            for block in merged.get("blocks", []) if block.get("type") == "table"
        }
        fingerprints = {block_fingerprint(block) for block in merged.get("blocks", [])}
        for document in documents[1:]:
            for block in document.get("blocks", []):
                if block.get("type") == "table":
                    key = tuple(block.get("headers", []))
                    if key in table_by_headers:
                        merge_table_blocks(table_by_headers[key], block)
                        continue
                fingerprint = block_fingerprint(block)
                if fingerprint not in fingerprints:
                    fingerprints.add(fingerprint)
                    merged["blocks"].append(block)

    for field, keys in [
        ("actions", ("url", "label")), ("attachments", ("attachment_id",)),
        ("images", ("url",)), ("links", ("url", "anchor_text")),
    ]:
        combined: list[dict[str, Any]] = []
        for document in documents:
            combined.extend(document.get(field, []))
        merged[field] = unique_dicts(combined, keys)

    apply_document_filters(merged)
    refresh_document(merged)
    return merged


def collect_paginated_document(
    session: requests.Session,
    first_result: FetchResult,
    first_document: dict[str, Any],
    row: dict[str, str],
    paths: OutputPaths,
    config: PipelineConfig,
) -> tuple[dict[str, Any], dict[str, Any]]:
    plan = detect_pagination_plan(first_result.body, first_result.encoding, first_result.final_url, row, config)
    if plan is None:
        collection = {
            "collection_scope": "single_public_page",
            "is_complete": True,
            "pagination_detected": False,
            "expected_page_count": 1,
            "fetched_page_count": 1,
            "failures": [],
        }
        first_document["pagination_collection"] = collection
        return first_document, collection

    if plan.last_internal_index + 1 > config.pagination_max_pages:
        raise ValueError(
            f"{row['url_id']} 페이지 수가 안전 한도 초과: {plan.last_internal_index + 1}"
        )

    page_dir = paths.pagination / row["url_id"]
    page_dir.mkdir(parents=True, exist_ok=True)
    first_page_path = page_dir / "page_001.html"
    first_page_path.write_bytes(first_result.body)

    page_documents = [first_document]
    page_records: list[dict[str, Any]] = [{
        "internal_page_index": 0 if plan.index_base == 0 else 1,
        "displayed_page_number": displayed_page_number(first_result.body, first_result.encoding),
        "status_code": first_result.status_code,
        "request_method": "GET",
        "final_url": first_result.final_url,
        "signature": repeatable_signature(first_document),
        "raw_sha256": first_result.raw_sha256,
        "raw_html_path": str(first_page_path.relative_to(paths.root)),
    }]
    failures: list[dict[str, Any]] = []
    seen_signatures = {page_records[0]["signature"]: page_records[0]["internal_page_index"]}

    if plan.index_base == 0:
        page_indexes = range(1, plan.last_internal_index + 1)
    else:
        page_indexes = range(2, plan.last_internal_index + 1)

    for page_index in page_indexes:
        try:
            started = time.perf_counter()
            if plan.method == "POST":
                payload = dict(plan.base_payload)
                payload[plan.page_param] = str(page_index)
                if plan.page_size_param:
                    payload[plan.page_size_param] = str(plan.page_size)
                response = session.post(
                    plan.endpoint, data=payload, headers={"Referer": first_result.final_url},
                    timeout=config.request_timeout_seconds, allow_redirects=True,
                )
                request_payload: dict[str, Any] = payload
            else:
                parsed = urlparse(plan.endpoint)
                query = parse_qs(parsed.query)
                query[plan.page_param] = [str(page_index)]
                query_string = urlencode(query, doseq=True)
                url = urlunparse(parsed._replace(query=query_string))
                response = session.get(url, timeout=config.request_timeout_seconds, allow_redirects=True)
                request_payload = {plan.page_param: page_index}
            response.raise_for_status()
            result = response_to_fetch_result(response, plan.endpoint, time.perf_counter() - started)
            page_document = parse_html_document(result.body, result.final_url, row, result.encoding)
            signature = repeatable_signature(page_document)
            page_number = page_index + 1 if plan.index_base == 0 else page_index
            page_path = page_dir / f"page_{page_number:03d}.html"
            page_path.write_bytes(result.body)
            record = {
                "internal_page_index": page_index,
                "displayed_page_number": displayed_page_number(result.body, result.encoding),
                "status_code": result.status_code,
                "request_method": plan.method,
                "request_payload": request_payload,
                "final_url": result.final_url,
                "signature": signature,
                "raw_sha256": result.raw_sha256,
                "raw_html_path": str(page_path.relative_to(paths.root)),
            }
            if signature in seen_signatures:
                failures.append({
                    "internal_page_index": page_index,
                    "reason": "다른 페이지와 같은 반복 콘텐츠",
                    "same_as": seen_signatures[signature],
                })
            else:
                seen_signatures[signature] = page_index
            expected_display = page_number
            actual_display = record["displayed_page_number"]
            if actual_display is not None and actual_display != expected_display:
                failures.append({
                    "internal_page_index": page_index,
                    "reason": "요청 페이지와 표시 페이지 불일치",
                    "expected": expected_display,
                    "actual": actual_display,
                })
            page_records.append(record)
            page_documents.append(page_document)
        except Exception as error:
            failures.append({
                "internal_page_index": page_index,
                "reason": "페이지 요청 또는 파싱 실패",
                "error_type": type(error).__name__,
                "error": str(error),
            })
        time.sleep(config.request_delay_seconds)

    expected_count = plan.last_internal_index + 1 if plan.index_base == 0 else plan.last_internal_index
    is_complete = not failures and len(page_documents) == expected_count
    merged = merge_paginated_documents(page_documents, row)
    collection = {
        "collection_scope": "all_public_pages" if is_complete else "partial_public_pages",
        "is_complete": is_complete,
        "pagination_detected": True,
        "detection_method": plan.detection_method,
        "method": plan.method,
        "endpoint": plan.endpoint,
        "page_param": plan.page_param,
        "index_base": plan.index_base,
        "expected_page_count": expected_count,
        "fetched_page_count": len(page_documents),
        "pages": page_records,
        "failures": failures,
        "collected_at": now_kst_iso(),
    }
    merged["pagination_collection"] = collection
    merged["dynamic_collection"] = collection if row["url_id"] == "BI-004" else None
    return merged, collection


# ============================================================
# 자산 다운로드
# ============================================================

def download_direct_assets(
    session: requests.Session,
    document: dict[str, Any],
    row: dict[str, str],
    paths: OutputPaths,
    config: PipelineConfig,
) -> dict[str, list[dict[str, Any]]]:
    result = {"attachments": [], "images": [], "failures": []}
    base_dir = paths.raw_assets / safe_name(row["business_function"]) / row["url_id"]

    if config.download_direct_attachments:
        for attachment in document.get("attachments", []):
            if attachment.get("download_method") != "href" or not attachment.get("url"):
                continue
            try:
                response = session.get(attachment["url"], timeout=config.request_timeout_seconds, stream=True)
                response.raise_for_status()
                filename = Path(urlparse(response.url).path).name or attachment["display_name"]
                output = base_dir / "attachments" / safe_name(filename)
                ensure_parent(output)
                written = 0
                with output.open("wb") as file:
                    for chunk in response.iter_content(64 * 1024):
                        if not chunk:
                            continue
                        written += len(chunk)
                        if written > config.max_asset_bytes:
                            raise ValueError("첨부파일 제한 용량 초과")
                        file.write(chunk)
                attachment.update({
                    "download_status": "downloaded",
                    "local_path": str(output.relative_to(paths.root)),
                    "size_bytes": written,
                    "sha256": sha256_bytes(output.read_bytes()),
                })
                result["attachments"].append(attachment)
            except Exception as error:
                result["failures"].append({
                    "type": "attachment", "name": attachment.get("display_name", ""),
                    "error": f"{type(error).__name__}: {error}",
                })

    if config.download_images:
        for image in document.get("images", []):
            if image.get("image_role") == "decorative":
                continue
            try:
                response = session.get(image["url"], timeout=config.request_timeout_seconds, stream=True)
                response.raise_for_status()
                filename = image.get("filename") or Path(urlparse(response.url).path).name
                output = base_dir / "images" / safe_name(filename)
                ensure_parent(output)
                written = 0
                with output.open("wb") as file:
                    for chunk in response.iter_content(64 * 1024):
                        if not chunk:
                            continue
                        written += len(chunk)
                        if written > config.max_asset_bytes:
                            raise ValueError("이미지 제한 용량 초과")
                        file.write(chunk)
                result["images"].append({
                    **image, "local_path": str(output.relative_to(paths.root)),
                    "size_bytes": written, "sha256": sha256_bytes(output.read_bytes()),
                })
            except Exception as error:
                result["failures"].append({
                    "type": "image", "url": image.get("url", ""),
                    "error": f"{type(error).__name__}: {error}",
                })
    return result


def ensure_playwright() -> None:
    try:
        import playwright  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "playwright"], check=True)
    subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)


def download_js_attachments(
    documents: list[dict[str, Any]], paths: OutputPaths, config: PipelineConfig
) -> list[dict[str, Any]]:
    records = []
    for document in documents:
        for attachment in document.get("attachments", []):
            if attachment.get("download_method") == "gfn_downloadFile":
                records.append((document, attachment))
    if not records or not config.download_js_attachments_with_playwright:
        return []
    ensure_playwright()
    from playwright.sync_api import sync_playwright
    results: list[dict[str, Any]] = []
    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        context = browser.new_context(accept_downloads=True, user_agent=config.user_agent, locale="ko-KR")
        grouped: dict[str, list[tuple[dict[str, Any], dict[str, Any]]]] = {}
        for document, attachment in records:
            grouped.setdefault(document["source_url"], []).append((document, attachment))
        for source_url, items in grouped.items():
            page = context.new_page()
            try:
                page.goto(source_url, wait_until="domcontentloaded", timeout=config.request_timeout_seconds * 1000)
                page.wait_for_function("typeof gfn_downloadFile === 'function'", timeout=config.request_timeout_seconds * 1000)
                for document, attachment in items:
                    try:
                        with page.expect_download(timeout=60_000) as info:
                            page.evaluate("([a,b]) => gfn_downloadFile(a,b)", [attachment["token1"], attachment["token2"]])
                        download = info.value
                        output = (
                            paths.raw_assets / safe_name(document["business_function"]) /
                            document["doc_id"] / "attachments" / safe_name(download.suggested_filename)
                        )
                        ensure_parent(output)
                        download.save_as(str(output))
                        attachment.update({
                            "download_status": "downloaded",
                            "filename": download.suggested_filename,
                            "local_path": str(output.relative_to(paths.root)),
                            "size_bytes": output.stat().st_size,
                            "sha256": sha256_bytes(output.read_bytes()),
                        })
                        results.append(attachment)
                    except Exception as error:
                        attachment.update({"download_status": "failed", "error": f"{type(error).__name__}: {error}"})
            finally:
                page.close()
        context.close()
        browser.close()
    return results


# ============================================================
# 문서 저장과 청킹
# ============================================================

def save_document(paths: OutputPaths, document: dict[str, Any], row: dict[str, str]) -> tuple[Path, Path]:
    folder = safe_name(row["business_function"])
    md_path = paths.parsed_markdown / folder / f"{row['url_id']}.md"
    json_path = paths.parsed_json / folder / f"{row['url_id']}.json"
    ensure_parent(md_path)
    md_path.write_text(document.get("content_markdown", ""), encoding="utf-8")
    write_json(json_path, document)
    return md_path, json_path


def render_table_chunk(headers: list[str], rows: list[list[str]]) -> str:
    return render_blocks([{"type": "table", "headers": headers, "rows": rows}])


def split_table(block: dict[str, Any], max_chars: int, heading_prefix: str = "") -> list[str]:
    headers, rows = block.get("headers", []), block.get("rows", [])
    prefix = f"### {heading_prefix}\n\n" if heading_prefix else ""
    complete = prefix + render_table_chunk(headers, rows)
    if len(complete) <= max_chars:
        return [complete]
    chunks: list[str] = []
    current: list[list[str]] = []
    for row in rows:
        candidate = prefix + render_table_chunk(headers, current + [row])
        if current and len(candidate) > max_chars:
            chunks.append(prefix + render_table_chunk(headers, current))
            current = [row]
        else:
            current.append(row)
    if current:
        chunks.append(prefix + render_table_chunk(headers, current))
    return chunks


def meaningful_chunk(text: str, min_chars: int) -> bool:
    plain = re.sub(r"[#|\-\s]", "", text)
    if len(plain) < min_chars:
        return False
    if any(fragment in text for fragment in NOISE_CONTAINS_TEXTS):
        return False
    return True


def structure_aware_chunks(document: dict[str, Any], config: PipelineConfig) -> list[dict[str, Any]]:
    if not document.get("indexable", True):
        return []
    intermediate: list[dict[str, str]] = []
    current_heading = ""
    current_parts: list[str] = []

    def flush() -> None:
        nonlocal current_parts
        content = "\n\n".join(part for part in current_parts if part).strip()
        current_parts = []
        if content and meaningful_chunk(content, config.min_chunk_chars):
            intermediate.append({"section_title": current_heading, "chunk_type": "section", "content": content})

    for block in document.get("blocks", []):
        kind = block.get("type")
        if kind == "heading":
            # 제목만 바로 저장하지 않고 다음 내용과 결합합니다.
            flush()
            current_heading = block.get("text", "")
            current_parts = [render_blocks([block])]
            continue
        if kind == "faq":
            flush()
            content = render_blocks([block])
            if meaningful_chunk(content, config.min_chunk_chars):
                intermediate.append({"section_title": block.get("question", ""), "chunk_type": "faq", "content": content})
            continue
        if kind == "table":
            # 직전 제목을 표 청크에 포함하고 제목 단독 청크는 만들지 않습니다.
            pending_heading = current_heading
            current_parts = []
            for content in split_table(block, config.chunk_max_chars, pending_heading):
                if meaningful_chunk(content, config.min_chunk_chars):
                    intermediate.append({"section_title": pending_heading, "chunk_type": "table", "content": content})
            continue
        block_text = render_blocks([block])
        candidate = "\n\n".join(current_parts + [block_text]).strip()
        if current_parts and len(candidate) > config.chunk_max_chars:
            flush()
            if current_heading:
                current_parts = [f"### {current_heading}"]
        current_parts.append(block_text)
    flush()

    records: list[dict[str, Any]] = []
    seen_hashes: set[str] = set()
    action_types = sorted({action.get("action_type", "") for action in document.get("actions", []) if action.get("action_type")})
    for item in intermediate:
        content = item["content"].strip()
        content_hash = sha256_text(content)
        if content_hash in seen_hashes:
            continue
        seen_hashes.add(content_hash)
        records.append({
            "chunk_id": f"{document['doc_id']}_chunk_{len(records):03d}",
            "parent_doc_id": document["doc_id"],
            "chunk_index": len(records),
            "title": document["title"],
            "section_title": item["section_title"],
            "chunk_type": item["chunk_type"],
            "business_function": document["business_function"],
            "target_business_function": document["target_business_function"],
            "page_type": document["page_type"],
            "rag_index_mode": document.get("rag_index_mode", "full"),
            "source_url": document["source_url"],
            "available_action_types": action_types,
            "content": content,
            "content_hash": content_hash,
        })
    return records


def build_link_only_document(row: dict[str, str]) -> dict[str, Any]:
    expected = [value.strip() for value in row.get("expected_action_types", "").split("|") if value.strip()]
    action_type = expected[0] if expected else "related_service"
    action = {
        "action_id": sha256_text(f"{row['url_id']}|{row['source_url']}")[:16],
        "label": row["page_title"],
        "url": row["source_url"],
        "action_type": action_type,
        "auth_required": row.get("action_auth", ""),
        "official_domain": allowed_action_domain(row["source_url"]),
        "requires_review": False,
    }
    return {
        "doc_id": row["url_id"], "parent_doc_id": row["url_id"],
        "record_type": "link_only", "indexable": False, "rag_index_mode": "none",
        "title": row["page_title"], "source_url": row["source_url"],
        "site_name": row["site_name"], "business_function": row["business_function"],
        "target_business_function": row["target_business_function"],
        "page_type": row["page_type"], "content_markdown": "", "content_text": "",
        "blocks": [], "attachments": [], "images": [], "links": [], "actions": [action],
        "policy": {
            "normalized_decision": row["normalized_decision"],
            "content_scope": row["content_scope"], "action_policy": row["action_policy"],
            "auth_required": row["auth_required"],
        },
        "quality": {"status": "pass", "reasons": [], "action_count": 1},
        "parsed_at": now_kst_iso(),
    }


# ============================================================
# 전체 실행
# ============================================================

def run_pipeline(
    review_csv_path: str | Path,
    runtime_root: str | Path,
    config: PipelineConfig | None = None,
) -> dict[str, Any]:
    config = config or PipelineConfig()
    runtime_root = Path(runtime_root)
    paths = create_output_paths(runtime_root)
    session = create_session(config)

    review_df = pd.read_csv(review_csv_path, encoding="utf-8-sig", dtype=str).fillna("")
    manifest_df = normalize_review_manifest(review_df)
    manifest_df.to_csv(paths.manifest / "manifest_full_42.csv", index=False, encoding="utf-8-sig")
    review_df.to_csv(paths.manifest / "review_policy_42.csv", index=False, encoding="utf-8-sig")

    target_df = manifest_df.copy()
    if config.select_business_functions:
        target_df = target_df[target_df["business_function"].isin(config.select_business_functions)]
    if config.run_only_url_ids:
        unknown = sorted(set(config.run_only_url_ids) - set(manifest_df["url_id"]))
        if unknown:
            raise ValueError(f"Manifest에 없는 URL ID: {unknown}")
        target_df = target_df[target_df["url_id"].isin(config.run_only_url_ids)]
    if config.max_urls is not None:
        target_df = target_df.head(config.max_urls)
    target_df.to_csv(paths.manifest / "manifest_run.csv", index=False, encoding="utf-8-sig")

    documents: list[dict[str, Any]] = []
    run_results: list[dict[str, Any]] = []
    crawl_jsonl = paths.logs / "crawl_results.jsonl"

    for _, series in tqdm(target_df.iterrows(), total=len(target_df), desc="KDIC 42개 정책 기반 수집"):
        row = row_to_dict(series)
        started = now_kst_iso()
        try:
            if row["normalized_decision"] == "link_only":
                document = build_link_only_document(row)
                save_document(paths, document, row)
                documents.append(document)
                result = {
                    "url_id": row["url_id"], "business_function": row["business_function"],
                    "page_title": row["page_title"], "status": "link_metadata_created",
                    "status_code": None, "content_chars": 0, "faq_count": 0,
                    "table_count": 0, "action_count": 1, "attachment_count": 0,
                    "pagination_detected": False, "pagination_is_complete": True,
                    "pagination_page_count": 0, "error_type": "", "error": "",
                }
            else:
                first = fetch_url(session, row["source_url"], config)
                html_path, meta_path = save_fetch_result(paths, row, first)
                if not first.ok:
                    raise requests.HTTPError(f"HTTP {first.status_code}: {first.final_url}")
                if "html" not in first.content_type.lower():
                    raise ValueError(f"HTML 응답이 아님: {first.content_type}")
                first_document = parse_html_document(first.body, first.final_url, row, first.encoding)
                document, pagination = collect_paginated_document(
                    session, first, first_document, row, paths, config
                )
                # 페이지 병합 후 품질을 다시 계산합니다.
                document["quality"] = build_quality(document, document.get("content_text", ""), row)
                assets = download_direct_assets(session, document, row, paths, config)
                document["downloaded_assets"] = assets
                md_path, json_path = save_document(paths, document, row)
                documents.append(document)
                result = {
                    "url_id": row["url_id"], "business_function": row["business_function"],
                    "page_title": document["title"], "status": (
                        "paginated_full_collection_created" if pagination.get("pagination_detected") and pagination.get("is_complete")
                        else "pagination_collection_incomplete" if pagination.get("pagination_detected")
                        else "parse_success"
                    ),
                    "status_code": first.status_code,
                    "content_chars": len(document.get("content_text", "")),
                    "quality_status": document.get("quality", {}).get("status", ""),
                    "quality_reasons": " | ".join(document.get("quality", {}).get("reasons", [])),
                    "retention_ratio": document.get("quality", {}).get("retention_ratio", 0),
                    "faq_count": document.get("quality", {}).get("faq_count", 0),
                    "table_count": document.get("quality", {}).get("table_count", 0),
                    "action_count": len(document.get("actions", [])),
                    "attachment_count": len(document.get("attachments", [])),
                    "downloaded_attachment_count": len(assets.get("attachments", [])),
                    "image_count": len(document.get("images", [])),
                    "asset_failure_count": len(assets.get("failures", [])),
                    "pagination_detected": pagination.get("pagination_detected", False),
                    "pagination_is_complete": pagination.get("is_complete", True),
                    "pagination_page_count": pagination.get("fetched_page_count", 1),
                    "pagination_failure_count": len(pagination.get("failures", [])),
                    "raw_html_path": str(html_path.relative_to(paths.root)),
                    "response_meta_path": str(meta_path.relative_to(paths.root)),
                    "parsed_markdown_path": str(md_path.relative_to(paths.root)),
                    "parsed_json_path": str(json_path.relative_to(paths.root)),
                    "error_type": "", "error": "",
                }
        except Exception as error:
            result = {
                "url_id": row["url_id"], "business_function": row["business_function"],
                "page_title": row["page_title"], "status": "failed", "status_code": None,
                "content_chars": 0, "faq_count": 0, "table_count": 0, "action_count": 0,
                "attachment_count": 0, "pagination_detected": False,
                "pagination_is_complete": False, "pagination_page_count": 0,
                "error_type": type(error).__name__, "error": str(error),
            }
        result["started_at"] = started
        result["finished_at"] = now_kst_iso()
        run_results.append(result)
        append_jsonl(crawl_jsonl, result)
        if row["normalized_decision"] != "link_only":
            time.sleep(config.request_delay_seconds)

    # JS 첨부파일은 전체 페이지 파싱 후 실행합니다.
    js_downloads = download_js_attachments(documents, paths, config)
    if js_downloads:
        # 다운로드 상태가 갱신됐으므로 JSON을 다시 저장합니다.
        by_id = {row["url_id"]: row_to_dict(row) for _, row in target_df.iterrows()}
        for document in documents:
            save_document(paths, document, by_id[document["doc_id"]])

    results_df = pd.DataFrame(run_results)
    results_df.to_csv(paths.logs / "crawl_results.csv", index=False, encoding="utf-8-sig")

    documents_path = paths.processed / "documents.jsonl"
    with documents_path.open("w", encoding="utf-8") as file:
        for document in documents:
            file.write(json.dumps(document, ensure_ascii=False) + "\n")

    chunks: list[dict[str, Any]] = []
    if config.create_chunks:
        for document in documents:
            chunks.extend(structure_aware_chunks(document, config))
    chunks_path = paths.processed / "chunks.jsonl"
    with chunks_path.open("w", encoding="utf-8") as file:
        for chunk in chunks:
            file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    # 품질 보고서
    quality_rows: list[dict[str, Any]] = []
    for document in documents:
        quality = document.get("quality", {})
        pagination = document.get("pagination_collection", {}) or {}
        quality_rows.append({
            "url_id": document["doc_id"], "business_function": document["business_function"],
            "title": document["title"], "indexable": document.get("indexable", True),
            "rag_index_mode": document.get("rag_index_mode", ""),
            "quality_status": quality.get("status", ""),
            "quality_reasons": " | ".join(quality.get("reasons", [])),
            "content_chars": len(document.get("content_text", "")),
            "faq_count": quality.get("faq_count", 0), "table_count": quality.get("table_count", 0),
            "action_count": len(document.get("actions", [])),
            "attachment_count": len(document.get("attachments", [])),
            "pagination_detected": pagination.get("pagination_detected", False),
            "pagination_is_complete": pagination.get("is_complete", True),
            "pagination_page_count": pagination.get("fetched_page_count", 1),
            "excluded_content_count": len(document.get("excluded_content", [])),
        })
    quality_df = pd.DataFrame(quality_rows)
    quality_df.to_csv(paths.quality / "quality_report.csv", index=False, encoding="utf-8-sig")

    # 회귀 검사
    by_id = {document["doc_id"]: document for document in documents}
    validations: list[dict[str, Any]] = []
    if "DP-001" in by_id:
        validations.append({"validation": "DP-001 1억원", "passed": "1억원" in by_id["DP-001"].get("content_text", "")})
    if "UN-003" in by_id:
        text = by_id["UN-003"].get("content_text", "")
        validations.append({"validation": "UN-003 미수령금 종류", "passed": all(k in text for k in ["예금보험금", "개산지급금", "파산배당금"])})
    if "BI-004" in by_id:
        pc = by_id["BI-004"].get("pagination_collection", {})
        validations.append({"validation": "BI-004 전체 페이지", "passed": pc.get("is_complete", False)})
    if "MT-009" in by_id:
        text = by_id["MT-009"].get("content_text", "")
        validations.append({"validation": "MT-009 구버전 금액 제외", "passed": "5천만원" not in text and "5천만 원" not in text})
    faq_ids = ["DP-005", "DP-006", "MT-004", "MT-005", "DA-008", "HP-002"]
    for doc_id in faq_ids:
        if doc_id in by_id:
            validations.append({
                "validation": f"{doc_id} FAQ 추출",
                "passed": count_block_type(by_id[doc_id].get("blocks", []), "faq") > 0,
            })
    validation_df = pd.DataFrame(validations)
    validation_df.to_csv(paths.quality / "regression_tests.csv", index=False, encoding="utf-8-sig")

    summary = {
        "run_id": paths.root.name,
        "manifest_count": len(manifest_df),
        "target_count": len(target_df),
        "document_count": len(documents),
        "chunk_count": len(chunks),
        "status_counts": results_df["status"].value_counts().to_dict() if not results_df.empty else {},
        "quality_counts": quality_df["quality_status"].value_counts().to_dict() if not quality_df.empty else {},
        "action_count": sum(len(document.get("actions", [])) for document in documents),
        "attachment_count": sum(len(document.get("attachments", [])) for document in documents),
        "pagination_complete_count": int(quality_df["pagination_is_complete"].fillna(False).sum()) if not quality_df.empty else 0,
        "regression_failed_count": int((~validation_df["passed"]).sum()) if not validation_df.empty else 0,
    }
    write_json(paths.quality / "quality_summary.json", summary)

    archive_path = Path(shutil.make_archive(
        base_name=str(paths.root), format="zip",
        root_dir=paths.root.parent, base_dir=paths.root.name,
    ))
    return {
        "paths": paths,
        "manifest_df": manifest_df,
        "target_df": target_df,
        "results_df": results_df,
        "quality_df": quality_df,
        "validation_df": validation_df,
        "documents": documents,
        "chunks": chunks,
        "summary": summary,
        "archive_path": archive_path,
    }


## 3. 42개 URL 검토표 자동 생성

In [ ]:
import base64
import gzip
from pathlib import Path

RUNTIME_ROOT = (
    Path("/content")
    if Path("/content").exists()
    else Path.cwd()
)

REVIEW_CSV_PATH = (
    RUNTIME_ROOT
    / "KDIC_42_URL_RAG_Action_검토표.csv"
)

EMBEDDED_REVIEW_GZIP_B64 = """H4sIAHLiVWoC/81cbW8T2dn+vtL+h1E+VESajeO8AFupW7HkYTcqFDZA1W+WY0+Cy2TsnRmH5dF+cIKDDAkltHFjgp2abgLJyjx1jJMYNatK/BzPGfUvPPd9nzPjGY8T4sSBSojY4zNnjs91v1z3y/F//vVvq5azCw3ZqjRYthgZH5PZ6oJVqUesp1lre4WV4KOft+1nRe8FGmv9vCDbfy6wUp29zkRYsWwX8rJdyLL1SoTlCuz1s2atysp5ubm3xNY3I2y3yP6x4FyjPxHrbaZZ+0WeuPQNvGzAvBGY33qywxZ3ZPdVxKqtsGJWvhQzE0ktYr2es+cqzqzi4q37KcUZADeyVyXZWtxgG888K6Q1wVPZzgI8P2M/LMPQQrNWcd419w+aO1WZ7RVZrRC5PXH188/C8tiNLwYHwzIr5JqNHFtftt7WYb9YuQj7cchV/sbO0xvDjJqJWCQVnVbkhBZT03ElMpVWVf8b2D5Yo3zdd6/E90Ri5Syr1eU/yrKdr1uL7yS2+pCVlmRrL2fns2xtBa7vssWKxJ8v8e8hcZgGJFxh/uX7fettFjGkB7zfZw/mmm9hbKPKNjKsBK92K+wloDLHigfWyyI8vM5e1gfkocGh818MXvgiPCIx2NJyUeI7KX1769pViW+dfMc0U8avQ6F79+4N3I0nYgNJfeCuHjJSoXjKMPWUnjRDN+C/m/cNE/9enTGNkKGoSsy8GdO1gXjy88+G+F4PdbXXna92uekdJ5Ga1aK1sAw7tf7MevqcvVi2XlWOgYW4GwWtVJDY6xziAK+spQyoURs4vd/d9l0d5rs6fAIJhhWDlEhfkQQVd+21JTZfwbfVIn7RE0m2mNM3I8tugDF5v4/bI/b3+jlc0k9LEmrxWh3k0fprsV+OJ+9pajIa92y35xVpS/PtgWTnt5r/gr1ffcYWMjBvvthsZKQOM0pWLWs/PkDxZ7Utay9jLy2x0oHELYlQBFbOgKXpIVQcIa863EzP6ITWCEdr5PRowWv7L7lTogVG452wGP5pASr2PCOQO46BAmjt1Q3ryYq1eSDxp8BY/6TW4hbYedry2ht7/t2ZbvktfRr+xmOtvR/lez/a1d73Nd+Wm9USyx7Y2Srs9pVL38H/1j8bsFCr/A7usB/u2vlttliGL3UOxB3ckf24IVl7WWuzgIjBDrJ3b2TnG1rzdba2zUqwu/RE/rj+Pnkq+v1xkMMVcKsD6C3ugos9FB8EJ1c4R3fQEMl1l6QtYPVou/tlz7q3V6xHdVgzKlrbkuuoXNZm3fpnhlAkJiHZD4rN/Yb1eEXquIU4EjYR7GWXeE8lNMMLeGwmNDnpeJUr0e9/r0dnzEsp9T6he56je/703gUx/rRgAJNaLEucJIB/mq9YLzO4qgHxLPD0kvV0wXr1C2nSHJjYByCiO2fgbmCf2z3OBb7TF06z0/y7nYMdYs8b/bKuTCm6osWUSCw5k4rqCSOpycoPtOPuzruDZEEricFUJUEkH29I12U+oUOUwLg1d5fBNlmvwS8QPZLcWVr3HWXYzoN5lRz6RfLOvf4cKz8Tvp4kP5+1FnNCL4Bzga/C3WYPHklsrcqyZZYrou0D42jN/x/bWwGNEMtzKMTqMq4SFXFte0C+FtUSU4phvt9vvoWHlzzCAnwdWFyI87kPYBpPhCb/NzWpaq6JnIzpN/C9w9Kuz+oGwXpR/nrcQ4URp9UCvIDvlgMD0N1lsJHcIrLFIjJP+KoYUmwWuqZufJZSQ+IRCIq+dz70Un5ekS3idnXBK8KDiLGPH3hoA9cifh1NIgyyft4S/A5AA6moQrTTQ9UbSxkz45PCtt1IGsZYbEYL8L4vOVxDp4WL9he2L8eqW4hax8/5xzIsL6YYRpe8+7CHtSCjy/1yNAVfWA7g4wznXMR+AQIgVMLjeSUeNqLmw6dnhgZsQDsQ4UGOxPDpkADlttYzvz1ZaOPOAuYC6OB8FT5sNh6JCxn3nRu1HHPrp6e0SDKlaLd11dGJ1WWQe4fX5XNgF+2nFTvbcMCFsJr0Zm8F4uyzwOF/fkipWgCEMAdhpFfWy3OJNq9DXCPH72vRGcBKTSbvplMuQOC/zETUD1gklZ5UEzEPbpwCgKyi8fGHSy+r8Ar2por2ZtQ1/AgXfdQviyd6EwRELpBElOecGdppRr8cS+s3QLAkq1qw3u5KN67fvMUjIN9i6MHE4wkqO19Ab5UwyDmriqn85paeVshO5pfIaZXn2HrlzOj814D6N4lZBSn9Fc0wOeJD8u3fk7/iDN3HyYUwHvWR75qwJLRtISHIX0n8bu/ArqMs34NJ94DP/WMBHBf8pedY1W2rlm/XSPiUFgPq63F2Qk9/5PD/qCtq1FTiEUPRZxNAiwAR8E7SIYEz6n1QnxE+fzRMiiswB4mArRExgTCxRHfKEB42zoBuEqcf175HY0uvv0nHg8o+zKEf6gH0Ph8C7u/hEhpQDH196LikxqGxkjAIgJv9fEVsrzAqJ81OHTGnIx9sPUe5vNovQCslkBbPUFDvpGZGY2ZHliNGUjYYXIJkKqoAFN9x/sr+VqfEGDFLHL1clMScKAP2g4z1pIpxYrZ64sRJe1SXjoaiAPY04jyG8oBafktRv0sbncEf4eAPfwLwTwSvfxGk+/SIwpZVAoqJqdrChr2S5e+kzlbBu1LHNvitQdB5++lR9Sl4sqz9t0dwO/doENSvvnF8hdDtzV7kxo6CmCu4Oa2jho+bevyaeSfeEedRjvPI2dl30DX28AluOve/GMutF/iorpX4iLnadBi01h3c5tADGAqXzCcgy+tUTYSFdtDwcgYULxTkEgTfEDDx+xxXTumG/GOIcs/AfH+rp5RvEwZa8ACe5+Vrt3h8Wd0GiWcPXyIywETs5wUHtO4+gq/CXiyLWB/g9IzwDhDZAGDX8mwiriSPjWmfk+onu9BWrQEFnkMcS5l209h3ZPWA7sKED0yC2Cxu0Pp+3ga9FHlyEd4IJi2yHL2yv0EY787oBv2H6hiA7QKHbehTwHaioNM3x6i1uIVlMH/IGfCiHaIg0KCM9XjzsDjUkx0V9kTYTURsL2u9qmD6+ywQ6hiDXuQoDX8ClDz+qnvm4/pAnKBymPdzx0mOB3SGF7PsnYcWx6KIgdw+XsIsVJARtY8SesddIRpR5ynrm2ACPC6yWZ2Df2dSRfJBjT4S/WMA7i853CMnhLtjTeMwhMU9mBBvz8gSmiIZe1QuXGgUouAJcNaXKTUBxHPLQRC+51QkdkeJ3ZUx4oG45/EG5tislaqF4ebihvUq20Um/db1G+FBUtWNZxDwWssFrFBBmIQs920ZBNEt4IKycqsA99JLgFWUOSjYablOcrt0l6hoidkp97RTPXnu9kMFj2tGXP9mdvKuW/MYGuRyMNpLOYA9k8KDxy2A0A5L/4XQ4xC+FAhxCV6eooe5+Eq9heSTlCM/hNatZCo8yGEKc5jO98A6+2yvX2lxrftbaM++ogpsqcDmwQu96doqu/N4Z/FYY9juWh1HAb/+aUmC9UA44dhkVn4G16mkk9biLZvMRx1lj70jKNPumc/rXQUN4+lg6y8kJSdK+7YjmNBD0XhcN5TQJfpzyTS1a6apBxpZhjieF84ATzcE5T7QjT19NQ83IO2Evhw1IWC/M6No5gkg9z4HFLWE5S7c3EY1GMV0XWkZoViGbgvUVqglCWKc8oZVfYEm9lI8OalIE0o0rujeAEgY2VMj7USkaFU53m6tpVM8OjTMQb/4XwX6CeH2T3FIae20dbWhwW7g9neQiAT1iXrUPgR2Sj8a6BEO9JcfEWg/5e4i7IlMJVRT0ZW4S8POiSwwZy39ch9v2ZR80Qt1n7idmIR6LQtQS+zFOztXFS2FWOTEj0DrUCic8X0njJ9GWQ0L8bgV1l93PA/lT0MfjaWk/BamHVVlOhq7H5lNGOmoKlmNFetlCVed0OLKD9FJVfnNlahqKOC6CxBr9VJCTN1AknUzpZudQq2hURIPYEgf0Q7UM1Z5C65Y5XeUyWvuF7HLFjdJjeiKkUzrMSVCe3OsEJlmAnpK0wjjSs8VRVRf29f1c+3jXyyDZPS7j3fbMlqhFVEx+8Ey8jKs2PKpOCRyc/cNCgdNyjNUnIJS3goY83xF8j0pkJwMsSrQt7LoU+5VmrJdCiYU9Wr0XnJiOq0GhIAns8LhjygE2ET3fJtly/aDkqxGtXhCm47oyTSof2fMnTqksAxOKpGwxbIPJS1hSonP6fmkZRqoT9Pl6k7N6v0+52r9cou1/8hpPKd9spN1puyLqBkHA/DXYJkWsM6wXXcyYujhX79uBV8ccAkD9bUqhikt0FEreq76N820eflOUOt5Giw89FGdQov/nrD04GXQ3AEAhGt1TEIuFptVbC776QBzG9623A5wd3T+AexdNnAs9Hn9UUrCyhOxu1jqugcGLHlvwAkZBu6AMGGkLmTg9sTVVoGyx7D/wTAnYqnOfIDn1sLDZxy9nT5u47kzqiECZOT/sZ2sWKY2X7a2bW2WsYKJ0deRmVAsEWGpyF9nAkYyDb7G6NChI/JndNbA0xIv+uZ4m9yAJPoaBblwRlN73OnhnCFmJwjeoZEbT5yFRz4Glh3SpFi0aVTd5jiOFVutUNn+eR7pfEIDgx4FiZ9VIp6ciJrQ7kaSGtCrNvMeiZJ2HGHl+7w6jfYTZIuHdmvbIiHlN/qiCREr0qIXBBkg1dMAkL42K+E1BP5nUW3EFRLkAezhE6ejrOQ0KvTJXFSxKob9ouvPcF2ephWQUyQlrbYU2Dbr749Ez6s/LycSbtiB+XjT4Q3UocLzA71KzbUEjksYpuS+U6di43HNvD9hqLxHZZjn58KjnywtTwLmMyRH53AdTSbu7p4sgu2080W/TTnUgnRwIEFfcTwX4fTdcomBiWvbTuNlyW1CF7lXALFY7pSupUpnj9ppOyXosV8hcCgnLI9d4sXOnSz2DOPBq7zbdPbBa+sVtI6uafF81uEA1AcQ9Xa4cXIdnK+N8QcwfTBnLdZbaMaSmpFWTQfKw9O1LShpO4YpFQ/+Cv0b4uM0j4HCIxGpwTP4YUDsSbEWKzQV79v2kH1xuoTq3/F0CuhD1FQi08CFU9TovZcBk9ozzPXJkDqpmtE/zZihq/DiErzA4NB93elQ1hD/wkO9xJ8iJafS/5+DhxKHRdiKaNq8o2gm7UWrJexQt3FUVCA6z/j0EncUnJtSWpZsSiBMcNbvudURF9GuJqRGbp2jI6tPpzklV4j6sV0NL4rrpTrFj54tcFfmOUnTqbvhI+A/rk0lx7Xvx4yYoaEtCMjBsBD8E8lBtwTwdxO/uuxTbsGxiL9TPrdZXcZYyz0XcdYKHyaFpwZ5AMdadk9Jcm2W8OJ8gZR2fouLPDjtZTocwUdwc0BF112gSdnTFVjjiZCuqEbCVEJ4EOJ3uhYDGKMAcTtwI/wLjPRGgWEb17ZBM8hT0VU5Gv9TNEZJ2yQYsPuyrswmlHtHuOW2CSTuD6mbC9MkpAvoon2NDs8bSC2F/As/WiuAkis/AM3UgD4mp6YSMeKRRPIdJXXubFQhzBZy1Mb4nQVhEf7FMrUfiefAVmTpNIu9miOS6RwtD/ERYlZBz7jPPhU9O6bKXtaV+ERsVr+PbwKQj3LIR3sCOWa556ugbHgavSuseecWvx/cIc3QOqssgnhEYjVHnZs+NuYgIKSApxKuyy7IlFWOp3XFaVA6FODSAfBJCZN0m2XRRgZA4Vm/dwtOdzCt1BnDTb5ohgBOCs5h2aXmNdBw/k14ThmjeFj9mSN+w9C+DkSBw+c50ud7grTAa20J4pETIU13ttSZHuWe/XPwbrfYHxVoN3CniB3PtYFntleWrPVCc78hCNnHwHJiOoDlBY7lheNjySMo2O5GjqcQMVYHzriTha+FxZEXyyFfEydPUQualeJ8E3+I4jQky6oW3VMleIIdsXCrrp4mqHae1bZQjM3aVyr5yjE/JgwjrbSTLlGGb5Eu92hWke0VsZaPy/OE/+0MyxNs+w5g9SLYDgZdBsCPeeUQysEVLT6mTJpjqjaNnKtTzm74IpeKi8eWis4tUR7WKdqCsS3maFcduOfY3TEeCu12BDsdxK2GiS7OEbca3GiWFod3yHVhA+u8WI1DKkVZn1ZDlAiwxUFj6qfxFFz4iZ+zanXykGsO6Jfyt+KHWkoZa/ERPBzcI4oeANf52mFqDl90Z4GtZxF//nMXpEH8I15pw8yXd04+JSdy2DZGUtJ1eja4SFf9n1bw51rwtJ74IHgoCK66aVlRthGSoiuppG620rPHoufuIyXMv8CKbo97yr28ycJtMadvTycIfIcFAkeHetMC2a7sl7XYzFhM1Tsp+sggl4uhY8tFZ0UPgH36HwBo73yDSR2setPh5pEa/BWPagYtzvomSAnVxDk1oGN83mZFAspxKD1ueUOkbugpE9Hi8IQ5PMOHwEO1KcpAgWPLgcijdEM4IXbrtDocnLNNh0XxrUXbHJfeZuINBTw6JpniyZloQmv9AAFPTTnhDSbbvc90ntUpxyZ12BEqAlLzeB+5+sBMxxGglo6bUX1aMSOTaSOhYSsJKJXbRfehvfcm2kAECwd2ftv5RQOwTr3kex1UfnzWmIqqtzVdHVen1UOVf4hL18ixlb+D7HCTaheyLZMqnfOenhEK4dRLQP/EUTvDbblyMrAtIigE5I+ybyZ+s2PGxTM9HMtztqeVyELzH1xjv+zae2dWnjfjZSTJz/f836fF+nymRZTfO6xY4jV654fDXlWwEAwEofzIfoxRnpOKdyZ2+kjaz6wQN+Q+okeyM5uMhfAEEohF6HJUVxR9QplOGKboE1Hh5eef/T9C2SPlBk8AAA=="""

REVIEW_CSV_PATH.write_bytes(
    gzip.decompress(
        base64.b64decode(
            EMBEDDED_REVIEW_GZIP_B64
        )
    )
)

print("42개 검토표 생성:", REVIEW_CSV_PATH)
print("파일 크기:", REVIEW_CSV_PATH.stat().st_size)

## 4. 실행 설정

### 처음 전체 실행 권장 설정

- 42개 전체 수집
- 공개 다중 페이지 자동 전체 수집
- 일반 URL 첨부파일 다운로드
- JavaScript 첨부파일은 우선 메타데이터만 수집
- 이미지 파일은 우선 다운로드하지 않음

JavaScript 기반 HWP·PDF 39개도 실제로 내려받으려면 다음 값을 `True`로 변경합니다.

```python
download_js_attachments_with_playwright=True
```

이 설정은 Chromium 설치와 버튼별 다운로드가 필요하여 실행 시간이 길어질 수 있습니다.

In [ ]:
from kdic_final_pipeline import (
    PipelineConfig,
    normalize_review_manifest,
    run_pipeline,
)

CONFIG = PipelineConfig(
    # 특정 업무만 실행할 때 사용
    select_business_functions=[],

    # 특정 문서만 시험할 때 사용
    # 예: ["BI-004", "DP-005", "BI-003"]
    run_only_url_ids=[],

    # 전체 실행은 None
    max_urls=None,

    request_delay_seconds=1.2,
    request_timeout_seconds=30,
    max_retries=3,

    # FAQ·목록·조회형 페이지네이션 전체 수집
    enable_generic_pagination=True,
    pagination_max_pages=100,
    pagination_page_size=10,

    # href로 직접 연결된 첨부파일
    download_direct_attachments=True,

    # 콘텐츠 이미지를 실제 파일로 내려받을지 여부
    download_images=False,

    # gfn_downloadFile 버튼을 Playwright로 실제 다운로드
    download_js_attachments_with_playwright=False,

    # RAG 구조 기반 청킹
    create_chunks=True,
    chunk_max_chars=1600,
    min_chunk_chars=80,
)

print(CONFIG)

## 5. 실행 전 자체 테스트

In [ ]:
import pandas as pd

from kdic_final_pipeline import (
    PipelineConfig,
    StructureParser,
    classify_action_type,
    count_block_type,
    detect_pagination_plan,
    extract_actions,
    normalize_review_manifest,
    parse_html_document,
)

review_df = pd.read_csv(
    REVIEW_CSV_PATH,
    encoding="utf-8-sig",
    dtype=str,
).fillna("")

manifest_df = normalize_review_manifest(
    review_df
)

assert len(manifest_df) == 42
assert manifest_df["url_id"].nunique() == 42

# FAQ 구조 테스트
faq_html = '''
<!doctype html>
<html lang="ko">
<body>
<div class="contents">
  <div class="accoWrap">
    <div class="accodian">
      <dl>
        <dt>질문 신청 기간은 언제인가요? 열기</dt>
        <dd><p>답변 공식 안내 기간에 신청합니다.</p></dd>
      </dl>
    </div>
  </div>
</div>
</body>
</html>
'''.encode("utf-8")

faq_row = {
    "url_id": "TEST-FAQ",
    "page_title": "FAQ 테스트",
    "source_url": "https://example.org/faq",
    "site_name": "테스트",
    "business_function": "테스트",
    "target_business_function": "테스트",
    "page_type": "faq",
    "parser_profile": "test",
    "normalized_decision": "include_full",
    "content_scope": "전체",
    "rag_index_label": "O",
    "rag_index_mode": "full",
    "action_policy": "X",
    "expected_action_types": "",
    "pagination_policy": "불필요",
    "auth_required": "해당 없음",
    "action_auth": "해당 없음",
}

faq_document = parse_html_document(
    faq_html,
    "https://example.org/faq",
    faq_row,
    "utf-8",
)

assert count_block_type(
    faq_document["blocks"],
    "faq",
) == 1

# Action 버튼 테스트
action_html = '''
<!doctype html>
<html lang="ko">
<body>
<div class="contents">
  <button onclick="gfn_openUrl(
    'https://fins.kdic.or.kr/mm/lgn/test.do'
  )">
    신청 바로가기
  </button>
  <p>공개 설명입니다.</p>
</div>
</body>
</html>
'''.encode("utf-8")

action_row = dict(faq_row)
action_row.update(
    {
        "url_id": "TEST-ACTION",
        "page_type": "static_page",
        "action_policy": "O(신청)",
        "expected_action_types": "apply",
    }
)

action_document = parse_html_document(
    action_html,
    "https://www.kdic.or.kr/test",
    action_row,
    "utf-8",
)

assert len(action_document["actions"]) == 1
assert (
    action_document["actions"][0]["action_type"]
    == "apply"
)

# 공통 페이지네이션 탐지 테스트
pagination_html = '''
<!doctype html>
<html lang="ko">
<body>
<form id="srchForm" method="post"></form>
<div class="paging">
  <strong title="현재 페이지"><span>1</span></strong>
  <a onclick="chgPagingNo(1);">2</a>
  <a onclick="chgPagingNo(2);">3</a>
  <a class="btnLast" onclick="chgPagingNo(2);">
    마지막 페이지
  </a>
</div>
<script>
function doSrchForm() {
  $("#srchForm").attr(
    "action",
    "/cm/bbs/selectFaq.do"
  );
}
function chgPagingNo(pageNo) {
  $("#srchForm").append(
    $("<input/>", {
      name: "curPage",
      value: pageNo
    })
  );
}
</script>
</body>
</html>
'''.encode("utf-8")

pagination_row = dict(faq_row)
pagination_row["pagination_policy"] = (
    "필수(FAQ 전체 페이지 자동수집)"
)

plan = detect_pagination_plan(
    pagination_html,
    "utf-8",
    "https://example.org/cm/bbs/selectFaq.do",
    pagination_row,
    PipelineConfig(),
)

assert plan is not None
assert plan.last_internal_index == 2
assert plan.page_param == "curPage"

print("Manifest 42개 검사: 통과")
print("FAQ 질문·답변 검사: 통과")
print("Action 링크 검사: 통과")
print("공통 페이지네이션 검사: 통과")

display(
    manifest_df[
        [
            "url_id",
            "business_function",
            "page_title",
            "normalized_decision",
            "rag_index_mode",
            "action_policy",
            "pagination_policy",
        ]
    ]
)

## 6. 42개 전체 크롤링 실행

처리 순서:

```text
42개 검토표
→ 실행 Manifest 변환
→ 공개 HTML 수집
→ 페이지네이션 자동 탐지
→ 전체 페이지 반복 요청
→ 페이지별 원본 저장
→ 본문·FAQ·표 병합
→ Action·첨부파일 추출
→ 품질 검사
→ documents.jsonl
→ chunks.jsonl
→ 결과 ZIP
```

In [ ]:
RESULT = run_pipeline(
    review_csv_path=REVIEW_CSV_PATH,
    runtime_root=RUNTIME_ROOT,
    config=CONFIG,
)

print("실행 완료")
print(
    RESULT["summary"]
)

display(
    RESULT["results_df"]
)

display(
    RESULT["quality_df"]
)

if not RESULT["validation_df"].empty:
    print("핵심 회귀 검사")
    display(
        RESULT["validation_df"]
    )

## 7. 핵심 결과만 확인

In [ ]:
results_df = RESULT["results_df"]
quality_df = RESULT["quality_df"]

print("수집 상태")
display(
    results_df.groupby(
        "status",
        dropna=False,
    )
    .size()
    .rename("count")
    .reset_index()
)

print("품질 상태")
display(
    quality_df.groupby(
        "quality_status",
        dropna=False,
    )
    .size()
    .rename("count")
    .reset_index()
)

print("페이지네이션이 탐지된 문서")
display(
    quality_df[
        quality_df["pagination_detected"]
        .fillna(False)
    ][
        [
            "url_id",
            "title",
            "faq_count",
            "table_count",
            "pagination_is_complete",
            "pagination_page_count",
        ]
    ]
)

print("Action 링크가 있는 문서")
display(
    quality_df[
        quality_df["action_count"] > 0
    ][
        [
            "url_id",
            "title",
            "action_count",
            "rag_index_mode",
        ]
    ]
)

print("사람 검수가 필요한 문서")
display(
    quality_df[
        quality_df["quality_status"].isin(
            ["review", "fail"]
        )
    ]
)

## 8. 결과 파일 위치

주요 결과:

```text
manifest/
├── review_policy_42.csv
├── manifest_full_42.csv
└── manifest_run.csv

raw/html/
raw/assets/
raw/response_meta/

diagnostics/pagination/
└── 문서_ID/
    ├── page_001.html
    ├── page_002.html
    └── ...

parsed/markdown/
parsed/json/

processed/
├── documents.jsonl
└── chunks.jsonl

quality/
├── quality_report.csv
├── quality_summary.json
└── regression_tests.csv

logs/
├── crawl_results.csv
└── crawl_results.jsonl
```

### Action 사용 방식

`documents.jsonl`의 각 문서에는 다음 구조가 포함됩니다.

```json
{
  "doc_id": "BI-003",
  "actions": [
    {
      "label": "예금보험금 신청 바로가기",
      "url": "공식 URL",
      "action_type": "apply",
      "auth_required": "필요"
    }
  ]
}
```

챗봇은 설명 청크를 검색한 뒤 `parent_doc_id`로 부모 문서의 Action을 찾아 답변 하단에 제공하면 됩니다.

## 9. 결과 ZIP 다운로드

In [ ]:
archive_path = RESULT["archive_path"]

print("결과 ZIP:", archive_path)
print(
    "크기:",
    f"{archive_path.stat().st_size / 1024 / 1024:.2f} MB",
)

try:
    from google.colab import files

    files.download(
        str(archive_path)
    )
except Exception as error:
    print(
        "자동 다운로드가 시작되지 않으면 "
        "왼쪽 파일 패널에서 직접 다운로드하세요."
    )
    print(error)

# 크롤링 방식 요약

이 코드는 URL을 모두 같은 방식으로 처리하지 않습니다.

## 일반 설명 페이지

```text
GET 요청
→ 원본 HTML 저장
→ 본문 구조 파싱
→ Markdown·JSON 생성
```

## FAQ 페이지

```text
첫 페이지 GET
→ 마지막 페이지 탐지
→ curPage 변경 POST 반복
→ 모든 페이지의 FAQ 질문·답변 추출
→ 중복 질문 제거
→ 하나의 문서로 병합
```

## 공개 목록·조회 페이지

```text
첫 화면 GET
→ 페이지 이동 방식 탐지
→ 전체 페이지 요청
→ 표 행 병합
→ 중복·반복·누락 검사
```

## 설명과 신청 버튼이 함께 있는 페이지

```text
공개 설명은 RAG 본문으로 저장
→ gfn_openUrl·a href 버튼 분석
→ 신청·조회·자가진단 Action 별도 저장
→ 로그인 이후 개인정보는 수집하지 않음
```

## 첨부파일 페이지

```text
gfn_downloadFile 버튼에서 파일 정보 추출
→ 문서와 첨부파일 연결
→ 설정을 켜면 Playwright로 실제 파일 다운로드
```

## RAG 청킹

```text
FAQ: 질문 + 답변
표: 제목 + 표 헤더 + 일부 행
일반 본문: 소제목 + 관련 문단
Action URL: 청크에 반복하지 않고 부모 문서에서 관리
```